# Principal Factor: A Training-Dataset Proposal for U.S. Data Center Siting

> **Project status:** A new, independent dataset proposal focused on the United States.  
> **Primary scope:** Power, water, and land resources as principal siting factors.  
> **Snapshot date:** August 22, 2026.  
> **Boundary of this version:** Establish the location inventory, feature dictionary, available data sources, and research basis without fabricating model-training results.

## Core conclusions

1. Public data can support site-level training features, but no ready-made dataset provides a nationally consistent ground-truth label for the “best” U.S. data center location.
2. Retain two parallel tasks: regression predicts a continuous recommendation score, `y_score` (higher is better), while ordinal multiclass classification predicts `y_grade ∈ {1,…,K}` (grade 1 is best and grade K is worst). Compare `K=3/4/5` as a hyperparameter.
3. Five factor models—power, water, network, land, and cost/market—produce continuous subscores and grade probabilities, and are compared with models trained directly on the full feature set.
4. Compare multiple machine-learning and deep-learning algorithms for each task. Select the best regression model and the best ordinal-classification model independently; they need not use the same algorithm.
5. Validate all conclusions with nested spatial cross-validation and an independent geographic test set. Hard constraints are non-compensatory: a triggered exclusion cannot be offset by a high model score.


# Part 1: Inferred Distribution of Independent U.S. Data Center Sites

The map uses `im3_us_data_center_locations.gpkg` from the IM3 Open Source Data Center Atlas. The study covers the 50 U.S. states and excludes the District of Columbia and Puerto Rico. The source contains three parallel layers—`point`, `building`, and `campus`—without a shared project-level primary key. Therefore, the 1,471 mapped records remaining after within-layer `id` deduplication cannot be interpreted directly as 1,471 independent campuses.

## Unit of analysis

The map uses a newly constructed `site_id` as its counting unit. Results are divided into three mutually exclusive types:

- **Explicit campus:** A campus from the original `campus` layer after deduplication by `id`.
- **Building-inferred site:** One or more buildings not assigned to an explicit campus and consolidated using spatial and identity evidence.
- **Standalone point:** A point that cannot be assigned to either an explicit campus or a building-inferred site. It represents a low-confidence independent location, not a confirmed campus.

## Decision evidence and boundaries

| Priority | Evidence | Default boundary | Treatment |
|---|---|---|---|
| 1 | Identical `id` within the same layer | Exact `id` match | Retain one record; this removes 3 duplicate campus records and 2 duplicate building records |
| 2 | A building centroid or point falls within a campus polygon | Boundary points count as inside | Assign to the explicit campus; spatial containment overrides name differences |
| 3 | A record is adjacent to a campus and shares identity information | Same name or `ref` and ≤250 m from the boundary; same operator and ≤100 m | Assign to the explicit campus |
| 4 | Unmatched buildings share consistent identity | Same name or `ref` and centroid distance ≤750 m; same operator and ≤250 m | Form building-inferred sites through connected-component clustering |
| 5 | A point matches a building-inferred site | Point inside a building; or same name/`ref` and ≤250 m; or same operator and ≤100 m | Assign the point to the inferred site |
| 6 | Insufficient spatial or identity evidence | Outside the preceding boundaries, or name, operator, and `ref` do not support a match | Do not force a merge; retain a single building or point as an independent site |

Distances are approximate surface distances calculated from WGS 84 coordinates. Building representative centroids, rather than complete parcel boundaries, are used; buildings crossing campus boundaries may therefore be misassigned. If a record matches multiple campuses, containment takes precedence, followed by the shortest distance.

Identity fields are lowercased and stripped of whitespace and punctuation before exact matching. Fuzzy name matching is not used, and records from different states are never merged. Building clustering is transitive within a state: if A matches B and B matches C, all three enter one inferred site even when the direct A-to-C distance exceeds the threshold.

## Default result and confidence

| Result type | Count | Confidence definition |
|---|---:|---|
| Explicit campus | 132 | High: the source provides a campus boundary |
| Building-inferred site | 731 | Medium for 116 sites formed from multiple buildings; low for 615 single-building sites |
| Standalone point | 88 | Low: only a location point is available, without a campus boundary |
| **Total inferred independent sites** | **951** | 132 high-confidence, 116 medium-confidence, and 703 low-confidence sites |

During entity resolution, 257 building records and 1 point were assigned to explicit campuses. Unmatched buildings formed 731 inferred sites, and another 14 points were assigned to these inferred sites.

## Boundary sensitivity

| Setting | Same-name/`ref` building distance | Same-operator building distance | Same-name/`ref` distance to campus | Same-operator distance to campus | Inferred sites |
|---|---:|---:|---:|---:|---:|
| Conservative merging | 500 m | 100 m | 100 m | 50 m | 1,069 |
| **Default boundary** | **750 m** | **250 m** | **250 m** | **100 m** | **951** |
| Permissive merging | 1,000 m | 500 m | 500 m | 250 m | 911 |

Thus, 951 is an entity-resolution estimate under explicit rules, not a manually verified government census. Wider boundaries merge more records and reduce the inferred site count. Before formal modeling, dense regions should undergo sampled manual review and threshold-sensitivity results should be reported.

- States with at least one inferred site: **45**
- Inferred independent sites across the 50 states under the default boundary: **951**
- States without source records are shown in gray; this does not imply that no data centers exist there.
- State counts, `site_id`, and confidence levels are used only for mapping, spatial grouping, and sample review; they are not model features in `X`.

Sources: [IM3 Atlas](https://github.com/IMMM-SFA/datacenter-atlas); state boundaries: [Census TIGERweb](https://tigerweb.geo.census.gov/arcgis/rest/services/TIGERweb/State_County/MapServer).


In [ ]:
# Reproduce the source-layer count with the Python standard library; run from the notebook directory
from pathlib import Path
import sqlite3
GPKG_PATH = Path('../data/raw/im3_us_data_center_locations.gpkg')
STATE_CODES = {'AL','AK','AZ','AR','CA','CO','CT','DE','FL','GA','HI','ID','IL','IN','IA','KS','KY','LA','ME','MD','MA','MI','MN','MS','MO','MT','NE','NV','NH','NJ','NM','NY','NC','ND','OH','OK','OR','PA','RI','SC','SD','TN','TX','UT','VT','VA','WA','WV','WI','WY'}
QUERY = '''SELECT state_abb,COUNT(DISTINCT id) FROM (
SELECT state_abb,id FROM point UNION ALL SELECT state_abb,id FROM building
UNION ALL SELECT state_abb,id FROM campus)
WHERE state_abb IS NOT NULL GROUP BY state_abb'''
with sqlite3.connect(GPKG_PATH) as con:
    STATE_COUNTS = {s:n for s,n in con.execute(QUERY) if s in STATE_CODES and n>0}
STATE_COUNTS = dict(sorted(STATE_COUNTS.items(), key=lambda x:-x[1]))
STATE_COUNTS


<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1000 590" style="max-width:100%;height:auto;background:white"><rect width="1000" height="590" fill="white"/><text x="500" y="28" text-anchor="middle" font-size="21" font-weight="700">Inferred Independent Data Center Sites by U.S. State (IM3 Entity Resolution)</text><text x="500" y="48" text-anchor="middle" font-size="11" fill="#6b7280">Darker colors indicate more inferred sites; gray indicates no records; default-boundary estimate, not an official census</text><path d="M 627.8,247.8 L 626.7,246.9 L 626.5,245.5 L 622.0,231.8 L 621.8,232.4 L 620.9,232.1 L 620.2,232.9 L 620.5,233.1 L 620.2,234.5 L 620.5,234.7 L 620.2,235.0 L 620.6,235.6 L 620.1,237.0 L 621.0,238.2 L 620.7,238.5 L 621.0,238.9 L 619.9,240.4 L 618.8,241.0 L 618.7,241.5 L 619.3,243.0 L 619.0,243.8 L 619.2,243.9 L 618.9,245.7 L 618.3,247.5 L 619.1,252.1 L 618.7,252.7 L 618.7,253.5 L 619.6,254.4 L 626.4,252.9 L 627.6,251.2 L 629.2,250.6 L 629.3,249.9 L 630.1,250.0 L 629.0,248.9 L 628.2,248.7 L 627.8,247.8 Z" fill="#fff7ec" stroke="#fff" stroke-width="0.7"><title>New Hampshire: 3 (explicit campuses 0; building-inferred sites 3; standalone points 0)</title></path><path d="M 334.6,266.0 L 336.3,259.6 L 311.1,252.4 L 311.0,253.6 L 310.4,253.2 L 310.0,253.6 L 311.0,255.1 L 310.8,257.2 L 310.7,257.6 L 310.0,257.7 L 310.3,258.3 L 309.4,259.4 L 309.3,260.9 L 306.3,264.5 L 306.3,266.3 L 307.4,268.0 L 308.2,270.7 L 308.1,272.7 L 307.2,274.2 L 307.3,276.8 L 306.8,277.4 L 306.8,278.0 L 309.4,283.1 L 309.7,285.3 L 309.0,286.7 L 310.1,286.9 L 311.5,288.9 L 311.0,291.2 L 311.3,292.6 L 310.9,294.0 L 311.2,295.0 L 312.7,296.8 L 312.6,299.0 L 312.2,299.5 L 312.1,302.0 L 313.2,303.6 L 313.4,305.0 L 314.5,307.1 L 314.6,308.4 L 315.3,308.9 L 316.0,310.5 L 316.6,310.9 L 316.1,312.0 L 316.3,312.6 L 317.6,313.7 L 316.9,315.2 L 317.1,315.8 L 316.3,317.8 L 316.4,318.3 L 317.1,318.8 L 317.6,319.7 L 319.6,320.0 L 322.3,321.4 L 323.2,321.4 L 324.6,322.8 L 325.3,324.7 L 327.4,326.0 L 329.2,326.1 L 329.6,327.4 L 329.2,328.3 L 330.1,329.1 L 331.4,329.1 L 334.9,334.0 L 335.3,336.7 L 334.9,338.7 L 335.4,339.1 L 335.4,339.9 L 353.8,341.9 L 354.3,341.3 L 354.5,340.2 L 354.2,339.7 L 353.3,339.5 L 353.1,339.0 L 353.6,337.6 L 353.3,337.2 L 353.6,336.8 L 353.5,336.4 L 354.0,336.4 L 355.0,335.4 L 355.2,334.4 L 355.5,334.2 L 355.6,332.3 L 356.4,331.7 L 356.5,331.2 L 358.8,330.2 L 358.6,329.4 L 357.4,328.3 L 357.3,326.1 L 356.4,324.6 L 356.7,323.6 L 342.1,301.7 L 330.1,283.5 L 334.6,266.0 Z M 307.9,287.9 L 308.4,289.2 L 309.1,289.1 L 309.2,288.7 L 308.6,287.8 L 307.9,287.9 Z M 324.5,329.0 L 324.6,329.9 L 325.5,329.6 L 325.3,329.0 L 324.5,329.0 Z M 327.4,329.9 L 328.2,331.9 L 329.2,332.4 L 329.8,332.2 L 329.9,331.6 L 329.4,330.6 L 328.4,329.8 L 327.4,329.9 Z M 326.5,333.5 L 327.4,336.1 L 328.6,336.1 L 328.7,335.7 L 327.5,333.4 L 326.9,333.2 L 326.5,333.5 Z M 323.0,324.6 L 323.7,325.0 L 324.4,324.6 L 322.8,323.5 L 322.1,323.6 L 320.5,322.7 L 319.9,323.1 L 318.2,322.7 L 317.3,321.6 L 316.2,321.8 L 316.5,322.9 L 317.7,323.4 L 318.6,324.7 L 320.2,324.3 L 321.5,324.8 L 323.0,324.6 Z M 320.6,330.0 L 320.7,330.6 L 321.2,331.1 L 322.3,330.9 L 321.5,329.8 L 320.6,330.0 Z" fill="#e34a33" stroke="#fff" stroke-width="0.7"><title>California: 81 (explicit campuses 3; building-inferred sites 69; standalone points 9)</title></path><path d="M 614.3,284.8 L 615.2,283.8 L 616.0,280.6 L 615.8,275.8 L 615.5,274.5 L 613.3,274.8 L 613.4,273.6 L 614.2,273.3 L 614.5,270.3 L 609.1,268.5 L 608.5,269.0 L 607.9,271.1 L 607.1,272.1 L 607.8,273.0 L 607.2,274.1 L 607.2,274.6 L 607.5,275.5 L 608.3,275.6 L 608.6,276.6 L 609.2,276.6 L 611.1,278.3 L 609.0,280.4 L 609.0,281.0 L 607.9,281.6 L 607.4,282.3 L 607.0,283.7 L 607.4,284.1 L 607.3,285.1 L 609.1,286.4 L 611.8,289.7 L 613.2,287.9 L 614.3,284.8 Z M 614.2,273.0 L 614.2,273.0 L 614.2,273.0 L 614.2,273.0 Z M 614.2,272.9 L 614.2,272.9 L 614.2,272.9 L 614.2,272.9 Z" fill="#fc8d59" stroke="#fff" stroke-width="0.7"><title>New Jersey: 45 (explicit campuses 4; building-inferred sites 38; standalone points 3)</title></path><path d="M 576.9,326.9 L 570.3,327.7 L 570.1,327.5 L 567.9,329.0 L 565.6,329.9 L 565.7,330.4 L 564.5,331.6 L 564.3,332.6 L 566.7,334.0 L 567.9,334.1 L 569.2,336.5 L 570.5,338.0 L 572.2,338.8 L 573.4,340.3 L 575.3,341.3 L 575.4,342.3 L 576.0,342.5 L 576.1,343.0 L 576.7,343.2 L 576.9,343.7 L 578.8,344.6 L 578.8,345.2 L 579.5,346.1 L 579.8,347.6 L 581.5,348.7 L 581.5,349.1 L 582.1,349.7 L 582.0,350.5 L 582.5,351.5 L 584.9,352.1 L 585.0,351.3 L 587.2,349.3 L 587.9,347.6 L 589.7,346.6 L 592.4,343.0 L 593.5,342.6 L 593.7,341.8 L 594.8,340.6 L 594.4,340.0 L 594.5,339.0 L 595.3,337.0 L 596.5,335.4 L 597.7,334.6 L 588.5,328.0 L 581.1,329.1 L 581.0,328.1 L 579.8,326.9 L 579.2,327.5 L 579.1,326.6 L 576.9,326.9 Z" fill="#fff7ec" stroke="#fff" stroke-width="0.7"><title>South Carolina: 3 (explicit campuses 2; building-inferred sites 1; standalone points 0)</title></path><path d="M 529.3,227.7 L 522.0,224.3 L 520.3,225.0 L 516.9,227.4 L 516.0,227.2 L 511.6,239.3 L 512.7,239.7 L 513.4,241.0 L 519.3,242.3 L 521.8,243.3 L 522.7,243.0 L 525.1,243.6 L 525.3,244.0 L 525.0,244.5 L 526.5,244.9 L 527.1,245.4 L 526.8,245.7 L 527.2,246.2 L 526.8,246.4 L 527.1,246.9 L 526.7,248.1 L 528.0,247.9 L 527.6,249.3 L 528.2,250.0 L 529.4,250.2 L 529.6,249.1 L 531.1,247.0 L 533.1,246.8 L 536.2,248.2 L 534.9,249.6 L 534.0,251.4 L 532.7,257.7 L 532.4,263.9 L 533.9,271.1 L 533.3,277.3 L 547.8,275.7 L 547.9,276.3 L 556.2,274.9 L 557.8,272.8 L 557.5,272.2 L 557.6,269.9 L 559.0,269.2 L 559.8,267.7 L 560.6,267.0 L 560.6,264.4 L 561.8,258.6 L 557.4,244.8 L 551.5,242.0 L 550.6,242.1 L 551.5,240.1 L 550.1,239.3 L 549.2,239.4 L 548.5,240.0 L 547.4,238.6 L 547.0,236.3 L 544.7,237.2 L 543.4,235.9 L 542.6,233.9 L 529.3,227.7 Z" fill="#fee8c8" stroke="#fff" stroke-width="0.7"><title>Michigan: 10 (explicit campuses 0; building-inferred sites 8; standalone points 2)</title></path><path d="M 514.0,342.1 L 514.3,341.7 L 515.0,341.7 L 514.2,341.4 L 514.1,341.0 L 514.7,341.3 L 514.7,340.7 L 515.4,340.6 L 515.2,340.0 L 515.7,339.9 L 515.8,340.4 L 515.9,339.8 L 516.3,339.5 L 516.2,338.9 L 516.6,338.3 L 516.2,337.8 L 516.3,337.2 L 516.6,337.7 L 517.0,337.2 L 516.3,337.0 L 516.4,336.4 L 516.8,336.5 L 516.6,336.9 L 517.1,336.8 L 516.8,335.6 L 517.1,335.6 L 517.1,336.1 L 518.0,335.8 L 518.4,335.2 L 518.0,334.2 L 518.6,334.3 L 518.8,333.5 L 519.4,333.3 L 519.1,332.9 L 519.2,332.5 L 518.7,332.2 L 519.2,331.3 L 518.6,331.3 L 519.0,330.5 L 519.4,331.2 L 519.7,330.9 L 519.5,329.9 L 520.3,330.1 L 519.9,329.6 L 520.6,328.9 L 520.1,329.0 L 519.8,328.4 L 521.5,327.5 L 521.0,327.2 L 521.1,326.8 L 521.8,326.9 L 521.2,326.1 L 516.9,326.4 L 518.9,323.7 L 518.8,323.0 L 518.3,322.8 L 518.2,322.1 L 489.0,323.2 L 490.4,332.4 L 490.3,346.9 L 491.0,347.6 L 491.9,347.2 L 491.9,347.5 L 492.3,347.5 L 492.3,347.2 L 493.3,347.5 L 493.4,351.9 L 513.0,351.4 L 513.3,350.9 L 512.7,350.4 L 513.5,350.2 L 513.4,349.6 L 513.7,349.1 L 513.3,349.4 L 513.0,348.5 L 513.6,347.7 L 513.0,348.3 L 512.6,348.1 L 513.1,347.8 L 513.0,347.4 L 512.4,347.8 L 512.7,347.3 L 512.3,346.8 L 513.0,346.4 L 512.3,345.8 L 513.6,345.8 L 513.0,345.5 L 512.9,345.0 L 513.9,344.9 L 513.3,344.3 L 513.7,343.6 L 513.1,343.2 L 513.9,343.4 L 513.8,342.9 L 514.5,342.8 L 514.5,342.3 L 514.0,342.1 Z" fill="#fff7ec" stroke="#fff" stroke-width="0.7"><title>Arkansas: 2 (explicit campuses 0; building-inferred sites 2; standalone points 0)</title></path><path d="M 532.2,356.5 L 532.7,334.4 L 531.9,333.6 L 517.9,334.6 L 518.4,335.1 L 518.0,335.8 L 517.1,336.1 L 517.1,335.6 L 516.8,335.6 L 516.9,336.9 L 516.4,336.4 L 516.3,337.0 L 517.0,337.2 L 516.6,337.7 L 516.3,337.2 L 516.2,337.8 L 516.6,338.3 L 516.2,338.9 L 516.3,339.6 L 515.9,339.8 L 515.8,340.4 L 515.7,339.9 L 515.2,340.0 L 515.4,340.6 L 514.7,340.7 L 514.7,341.3 L 514.1,341.1 L 515.0,341.7 L 513.9,342.0 L 514.6,342.4 L 514.4,342.8 L 513.8,342.9 L 514.0,343.4 L 513.1,343.1 L 513.2,343.5 L 513.7,343.6 L 513.3,344.4 L 513.9,344.9 L 512.9,344.9 L 512.8,345.4 L 513.6,345.8 L 512.3,345.8 L 513.0,346.4 L 512.3,346.8 L 512.7,347.3 L 512.4,347.8 L 513.0,347.4 L 513.1,347.8 L 512.6,348.1 L 513.0,348.3 L 513.6,347.7 L 513.1,348.9 L 513.4,349.4 L 513.7,349.1 L 513.4,349.6 L 513.5,350.2 L 512.7,350.4 L 513.3,350.9 L 512.7,352.0 L 513.2,352.2 L 513.2,351.5 L 513.6,351.6 L 513.8,352.2 L 513.2,352.7 L 513.1,353.4 L 513.9,353.7 L 513.3,354.5 L 513.5,354.8 L 514.2,354.3 L 513.8,355.0 L 514.5,355.5 L 513.7,355.1 L 513.6,355.6 L 514.6,355.9 L 514.5,356.7 L 515.3,356.5 L 515.0,357.1 L 514.6,357.1 L 514.6,358.0 L 514.2,357.6 L 513.4,358.0 L 513.4,358.5 L 514.2,358.6 L 514.1,358.1 L 514.5,358.4 L 514.0,359.2 L 513.5,359.0 L 514.1,359.5 L 513.3,359.9 L 513.4,360.3 L 512.8,360.8 L 512.9,361.2 L 512.6,360.8 L 512.3,361.0 L 512.2,361.7 L 512.9,361.7 L 512.2,361.8 L 512.0,362.8 L 511.2,362.8 L 512.0,363.2 L 511.2,363.7 L 511.6,364.9 L 511.1,364.4 L 510.8,364.6 L 511.3,365.7 L 510.4,365.9 L 510.8,366.4 L 510.6,367.0 L 511.1,367.5 L 510.6,368.0 L 523.9,367.2 L 523.2,370.1 L 523.6,371.0 L 524.5,371.7 L 525.4,373.9 L 528.2,373.5 L 530.3,373.9 L 530.8,373.2 L 533.9,373.5 L 532.1,359.3 L 532.2,356.5 Z" fill="#fff7ec" stroke="#fff" stroke-width="0.7"><title>Mississippi: 2 (explicit campuses 1; building-inferred sites 1; standalone points 0)</title></path><path d="M 496.2,289.3 L 481.4,289.5 L 481.5,289.9 L 482.1,289.8 L 481.9,290.1 L 482.3,291.5 L 482.1,291.7 L 483.3,292.3 L 483.2,292.7 L 483.8,293.3 L 483.7,294.0 L 485.5,295.3 L 486.7,295.2 L 487.0,295.8 L 486.7,296.1 L 487.1,296.4 L 486.5,296.4 L 485.6,298.0 L 486.6,299.3 L 487.0,299.3 L 486.9,299.9 L 487.4,300.7 L 488.9,301.3 L 489.0,323.2 L 518.2,322.1 L 518.3,322.8 L 518.8,323.0 L 518.9,323.7 L 516.9,326.4 L 521.3,326.0 L 521.5,325.3 L 522.0,324.8 L 521.2,324.0 L 522.3,323.9 L 521.7,323.3 L 522.4,322.9 L 522.0,321.3 L 522.5,321.3 L 522.7,322.2 L 523.2,320.7 L 524.2,321.1 L 524.5,320.3 L 524.2,319.7 L 524.8,319.5 L 524.3,319.0 L 524.8,317.8 L 524.2,317.8 L 523.7,317.0 L 523.3,317.1 L 523.6,317.7 L 522.9,317.3 L 522.3,315.6 L 521.9,315.4 L 522.4,314.5 L 521.7,313.3 L 522.0,312.8 L 521.6,312.0 L 520.7,311.6 L 520.6,311.2 L 519.4,310.4 L 518.8,310.6 L 518.6,310.2 L 518.9,309.9 L 517.7,309.5 L 516.0,308.0 L 515.8,307.0 L 516.9,304.7 L 516.7,303.8 L 517.2,302.7 L 515.1,301.8 L 514.4,302.7 L 513.7,302.2 L 513.1,299.5 L 508.9,295.8 L 507.9,292.6 L 508.0,291.3 L 508.4,290.5 L 507.7,290.2 L 507.7,289.9 L 506.4,288.6 L 496.2,289.3 Z" fill="#fee8c8" stroke="#fff" stroke-width="0.7"><title>Missouri: 13 (explicit campuses 1; building-inferred sites 10; standalone points 2)</title></path><path d="M 430.7,218.0 L 371.6,208.9 L 369.8,217.0 L 371.1,219.6 L 371.1,220.5 L 370.7,220.8 L 371.3,221.5 L 370.5,221.8 L 371.4,222.4 L 371.7,223.2 L 372.6,223.6 L 372.6,224.2 L 373.3,225.0 L 373.6,226.2 L 374.2,226.7 L 374.0,227.2 L 374.2,227.7 L 374.7,228.0 L 374.7,228.7 L 375.4,228.5 L 375.5,229.4 L 377.1,229.6 L 376.8,230.6 L 376.4,230.7 L 376.5,231.2 L 376.1,231.6 L 376.0,232.4 L 375.7,232.5 L 375.7,233.3 L 375.2,233.4 L 375.5,233.8 L 375.1,234.3 L 375.6,234.9 L 375.5,235.6 L 374.8,235.8 L 374.4,236.3 L 374.7,237.0 L 374.2,237.4 L 374.0,238.1 L 374.6,238.2 L 375.2,239.1 L 375.8,238.5 L 376.8,238.3 L 377.3,237.6 L 377.8,237.7 L 377.8,238.3 L 378.4,238.5 L 378.1,239.2 L 378.5,239.2 L 378.2,240.1 L 378.5,241.4 L 379.6,243.3 L 379.5,244.1 L 379.1,244.2 L 379.3,244.9 L 379.8,245.6 L 380.4,245.4 L 381.0,245.9 L 381.2,248.7 L 381.9,249.6 L 382.6,248.6 L 384.8,249.3 L 385.1,248.6 L 385.5,248.5 L 386.5,249.0 L 387.9,248.9 L 388.1,249.4 L 388.9,249.1 L 390.2,249.5 L 390.0,248.6 L 390.9,247.8 L 391.7,249.0 L 391.6,249.4 L 392.5,250.4 L 393.2,246.1 L 433.5,251.1 L 436.2,218.5 L 430.7,218.0 Z" fill="#fff7ec" stroke="#fff" stroke-width="0.7"><title>Montana: 2 (explicit campuses 0; building-inferred sites 2; standalone points 0)</title></path><path d="M 488.9,312.8 L 488.9,301.3 L 487.4,300.7 L 486.9,299.9 L 487.0,299.3 L 486.6,299.3 L 485.6,298.1 L 486.5,296.4 L 487.1,296.4 L 486.7,296.1 L 487.0,295.7 L 486.5,295.1 L 485.7,295.4 L 484.3,294.3 L 442.3,293.1 L 440.8,318.0 L 489.0,319.1 L 488.9,312.8 Z" fill="#fee8c8" stroke="#fff" stroke-width="0.7"><title>Kansas: 7 (explicit campuses 0; building-inferred sites 7; standalone points 0)</title></path><path d="M 541.6,306.2 L 542.0,306.5 L 542.1,307.2 L 543.7,307.9 L 544.3,307.3 L 544.3,306.0 L 544.7,305.1 L 545.7,304.8 L 546.0,303.6 L 547.1,302.7 L 546.8,301.1 L 547.8,300.9 L 548.5,301.3 L 549.6,300.4 L 550.7,300.2 L 550.8,299.4 L 550.2,299.2 L 550.4,298.7 L 549.9,298.0 L 550.3,297.6 L 547.8,275.7 L 531.4,277.4 L 533.1,297.3 L 532.6,297.7 L 533.0,298.4 L 532.5,299.1 L 533.6,300.6 L 533.4,301.3 L 533.8,302.3 L 533.1,303.2 L 533.2,303.6 L 532.9,304.0 L 533.0,304.3 L 532.4,304.7 L 532.5,305.2 L 532.0,306.3 L 531.8,306.1 L 531.0,306.6 L 531.6,307.3 L 531.0,307.9 L 531.4,308.0 L 530.8,308.3 L 531.1,309.0 L 530.8,309.4 L 531.1,309.6 L 530.6,309.6 L 531.1,310.1 L 530.7,310.2 L 531.7,310.5 L 531.9,310.2 L 531.6,309.6 L 531.9,309.2 L 532.3,309.6 L 533.3,309.3 L 533.3,309.9 L 533.7,309.9 L 533.8,308.7 L 534.3,309.2 L 535.1,308.9 L 537.0,310.0 L 537.4,308.9 L 538.7,308.0 L 539.3,308.8 L 539.8,308.7 L 540.0,309.2 L 540.2,308.5 L 540.7,308.4 L 540.5,307.5 L 541.1,307.2 L 540.8,306.8 L 541.7,306.6 L 541.3,306.2 L 541.6,306.2 Z" fill="#fff7ec" stroke="#fff" stroke-width="0.7"><title>Indiana: 5 (explicit campuses 0; building-inferred sites 3; standalone points 2)</title></path><path d="M 464.4,245.1 L 434.1,243.3 L 432.0,267.5 L 465.1,269.3 L 465.4,269.9 L 467.8,271.3 L 468.4,271.2 L 469.0,270.5 L 472.6,270.7 L 473.2,271.3 L 475.9,272.3 L 475.7,272.7 L 476.3,273.6 L 477.3,273.7 L 477.0,273.5 L 476.9,272.5 L 476.2,271.7 L 477.0,269.8 L 476.9,269.2 L 477.4,268.5 L 477.2,267.7 L 476.6,267.6 L 476.5,267.1 L 476.9,267.0 L 476.9,266.3 L 476.4,265.8 L 476.5,265.4 L 477.3,265.4 L 477.4,250.6 L 477.0,249.9 L 476.0,249.6 L 475.1,248.0 L 476.7,246.3 L 476.8,245.4 L 464.4,245.1 Z" fill="#fff7ec" stroke="#fff" stroke-width="0.7"><title>South Dakota: 2 (explicit campuses 0; building-inferred sites 2; standalone points 0)</title></path><path d="M 420.7,316.3 L 440.8,318.0 L 442.9,284.8 L 400.1,280.5 L 395.6,313.4 L 420.7,316.3 Z" fill="#fdbb84" stroke="#fff" stroke-width="0.7"><title>Colorado: 19 (explicit campuses 0; building-inferred sites 16; standalone points 3)</title></path><path d="M 606.4,282.0 L 607.5,282.1 L 609.0,281.0 L 609.0,280.4 L 611.1,278.3 L 609.2,276.6 L 608.6,276.6 L 608.3,275.6 L 607.5,275.5 L 607.2,274.6 L 607.2,274.1 L 607.8,273.0 L 607.1,272.1 L 607.9,271.1 L 608.5,269.0 L 609.1,268.5 L 608.7,268.0 L 607.1,267.9 L 606.3,267.0 L 606.2,265.8 L 605.9,265.7 L 606.0,265.4 L 605.3,265.0 L 604.8,265.2 L 604.1,264.3 L 577.8,269.4 L 577.1,265.2 L 575.4,266.5 L 572.8,267.5 L 576.3,288.7 L 605.5,283.2 L 606.4,282.0 Z" fill="#fee8c8" stroke="#fff" stroke-width="0.7"><title>Pennsylvania: 9 (explicit campuses 0; building-inferred sites 8; standalone points 1)</title></path><path d="M 353.3,229.8 L 361.6,231.8 L 361.4,230.9 L 361.9,230.4 L 361.4,228.7 L 366.3,207.8 L 333.2,199.1 L 334.4,200.9 L 334.3,201.4 L 332.7,201.6 L 332.9,203.9 L 331.9,204.8 L 329.5,204.5 L 324.5,200.7 L 324.4,201.2 L 323.9,201.4 L 324.0,202.3 L 323.2,203.0 L 323.3,203.9 L 323.0,204.7 L 323.7,207.2 L 323.5,207.7 L 323.9,208.0 L 323.6,210.1 L 323.8,211.2 L 323.8,212.5 L 323.4,213.9 L 323.3,216.0 L 322.3,219.2 L 323.0,219.3 L 323.1,219.1 L 323.8,219.8 L 325.9,220.2 L 326.3,221.3 L 327.8,221.5 L 328.8,222.8 L 328.9,224.5 L 328.5,226.2 L 328.9,226.6 L 331.0,227.8 L 333.5,227.3 L 335.4,227.5 L 337.1,228.5 L 337.3,229.1 L 337.9,228.8 L 338.8,229.2 L 340.5,228.7 L 341.2,229.4 L 342.8,229.5 L 344.3,229.1 L 346.0,229.2 L 346.7,228.8 L 349.2,229.4 L 350.1,229.0 L 353.3,229.8 Z" fill="#fc8d59" stroke="#fff" stroke-width="0.7"><title>Washington: 47 (explicit campuses 10; building-inferred sites 36; standalone points 1)</title></path><path d="M 502.4,351.7 L 493.4,351.9 L 493.6,360.4 L 494.5,361.2 L 495.1,362.2 L 495.3,362.7 L 495.1,363.7 L 495.9,364.3 L 495.7,364.7 L 496.5,365.4 L 496.2,366.0 L 496.6,366.3 L 496.9,367.1 L 497.3,367.0 L 497.1,367.7 L 497.5,368.2 L 497.0,368.5 L 497.4,369.0 L 497.0,369.4 L 497.2,369.9 L 495.9,372.3 L 496.2,373.1 L 495.8,374.0 L 496.2,374.4 L 496.4,375.6 L 496.1,376.0 L 496.3,376.3 L 494.7,378.3 L 495.6,380.1 L 496.3,379.3 L 499.8,378.9 L 504.1,380.4 L 506.5,380.7 L 508.0,380.2 L 509.5,381.5 L 511.1,381.3 L 512.8,381.9 L 513.7,382.7 L 516.0,383.2 L 515.7,383.8 L 516.4,384.2 L 519.1,383.7 L 520.6,383.9 L 522.8,382.7 L 525.8,382.0 L 526.8,382.7 L 527.0,384.6 L 527.4,384.6 L 528.6,383.5 L 529.1,384.0 L 529.5,383.9 L 530.7,382.2 L 530.8,381.5 L 529.5,380.9 L 529.0,379.9 L 531.1,377.3 L 531.3,375.1 L 530.6,373.9 L 528.2,373.5 L 525.9,374.0 L 525.2,373.6 L 524.5,371.7 L 523.6,371.0 L 523.2,370.1 L 523.9,367.2 L 510.6,368.0 L 511.1,367.5 L 510.6,367.0 L 510.8,366.4 L 510.4,365.9 L 511.3,365.7 L 510.8,364.6 L 511.1,364.4 L 511.6,364.9 L 511.2,363.7 L 512.0,363.2 L 511.2,362.8 L 512.0,362.8 L 512.2,361.8 L 512.9,361.7 L 512.2,361.7 L 512.3,361.0 L 512.6,360.8 L 512.9,361.2 L 512.8,360.8 L 513.4,360.3 L 513.3,359.9 L 514.1,359.5 L 513.5,359.0 L 514.0,359.2 L 514.5,358.4 L 514.1,358.1 L 514.2,358.6 L 513.4,358.5 L 513.4,358.0 L 514.2,357.6 L 514.6,358.0 L 514.6,357.1 L 515.0,357.1 L 515.3,356.5 L 514.5,356.7 L 514.6,355.9 L 513.6,355.6 L 513.7,355.1 L 514.5,355.5 L 513.8,355.0 L 514.2,354.3 L 513.5,354.8 L 513.3,354.5 L 513.9,353.7 L 513.1,353.4 L 513.2,352.7 L 513.8,352.2 L 513.6,351.5 L 513.3,351.5 L 513.0,352.2 L 512.7,352.0 L 513.0,351.4 L 502.4,351.7 Z" fill="#fff7ec" stroke="#fff" stroke-width="0.7"><title>Louisiana: 1 (explicit campuses 0; building-inferred sites 0; standalone points 1)</title></path><path d="M 639.3,223.6 L 639.0,222.8 L 639.2,222.4 L 638.8,222.1 L 639.0,221.7 L 636.1,212.8 L 633.0,211.2 L 632.3,211.5 L 632.4,212.0 L 631.3,212.3 L 629.8,213.7 L 628.9,213.3 L 628.5,212.0 L 627.5,212.0 L 625.0,219.2 L 625.3,221.5 L 624.7,222.2 L 624.5,223.6 L 624.9,223.9 L 624.7,225.0 L 625.0,225.1 L 624.9,225.4 L 625.3,225.5 L 625.3,225.9 L 624.7,226.9 L 624.9,227.5 L 624.2,228.2 L 623.6,229.6 L 624.4,230.5 L 623.4,230.4 L 623.5,232.1 L 622.7,231.3 L 622.0,231.8 L 626.5,245.5 L 626.7,246.9 L 628.0,247.9 L 628.2,248.7 L 629.0,248.9 L 630.1,250.0 L 630.1,249.2 L 629.6,248.9 L 629.6,248.6 L 629.8,248.2 L 630.4,248.6 L 630.6,248.1 L 630.3,247.8 L 629.8,248.0 L 629.6,247.2 L 630.4,246.4 L 631.1,244.5 L 633.5,242.2 L 634.2,242.1 L 634.3,241.4 L 634.7,241.0 L 635.4,241.8 L 635.8,241.2 L 635.6,240.8 L 636.2,240.7 L 636.6,240.1 L 637.6,240.7 L 638.2,240.7 L 638.8,239.3 L 638.3,238.7 L 639.1,238.5 L 639.7,237.5 L 640.9,236.7 L 640.9,235.3 L 641.6,235.0 L 641.9,234.4 L 642.5,234.3 L 642.5,233.6 L 643.7,233.3 L 644.2,232.3 L 645.2,231.4 L 646.5,229.2 L 644.1,226.6 L 643.4,226.6 L 643.2,227.2 L 642.0,226.2 L 642.2,225.4 L 641.4,224.6 L 641.9,224.4 L 641.6,223.8 L 639.3,223.6 Z" fill="#fff7ec" stroke="#fff" stroke-width="0.7"><title>Maine: 2 (explicit campuses 0; building-inferred sites 1; standalone points 1)</title></path><path d="M 602.2,239.3 L 600.8,239.7 L 599.3,241.2 L 596.6,245.9 L 594.6,247.7 L 594.0,248.7 L 592.9,252.9 L 581.8,255.0 L 579.0,257.0 L 580.2,258.4 L 580.3,259.9 L 580.7,259.9 L 580.8,260.5 L 581.6,261.2 L 581.2,262.0 L 577.1,265.2 L 577.8,269.4 L 604.0,264.2 L 604.8,265.2 L 605.3,265.0 L 606.0,265.4 L 605.9,265.7 L 606.2,265.8 L 606.3,267.0 L 607.1,267.9 L 608.7,268.0 L 609.1,268.5 L 614.5,270.3 L 614.2,273.3 L 613.4,273.7 L 613.3,274.8 L 613.5,275.0 L 619.1,273.0 L 626.9,266.9 L 626.7,265.7 L 625.6,264.9 L 616.3,270.3 L 615.9,270.1 L 615.3,269.3 L 616.5,268.0 L 616.0,267.4 L 614.7,261.0 L 614.8,255.4 L 613.4,249.0 L 612.9,248.4 L 612.5,248.5 L 612.5,249.0 L 612.2,248.9 L 612.3,247.3 L 611.3,245.2 L 611.3,244.0 L 611.6,243.3 L 611.3,242.5 L 611.4,241.9 L 610.5,240.6 L 610.5,239.1 L 610.1,238.7 L 610.0,237.4 L 602.2,239.3 Z M 614.2,272.9 L 614.2,272.9 L 614.2,272.9 L 614.2,272.9 Z M 614.2,273.0 L 614.2,273.0 L 614.2,273.0 L 614.2,273.0 Z" fill="#fdbb84" stroke="#fff" stroke-width="0.7"><title>New York: 20 (explicit campuses 0; building-inferred sites 16; standalone points 4)</title></path><path d="M 340.2,260.6 L 336.3,259.6 L 330.1,283.5 L 342.1,301.7 L 356.7,323.6 L 357.0,323.0 L 356.7,322.7 L 357.2,322.6 L 357.4,322.2 L 357.2,319.5 L 357.5,318.6 L 357.4,316.9 L 357.9,316.5 L 357.5,315.5 L 357.6,314.6 L 358.9,314.3 L 360.2,314.6 L 360.8,315.8 L 361.4,315.9 L 362.4,314.6 L 371.4,267.5 L 340.2,260.6 Z" fill="#fc8d59" stroke="#fff" stroke-width="0.7"><title>Nevada: 33 (explicit campuses 3; building-inferred sites 28; standalone points 2)</title></path><path d="M 257.7,475.8 L 260.7,480.0 L 267.5,485.0 L 266.8,485.5 L 266.9,485.8 L 269.0,486.5 L 269.9,485.5 L 270.0,486.0 L 270.8,486.0 L 270.6,486.6 L 271.0,486.8 L 270.5,487.8 L 271.1,488.0 L 271.1,488.6 L 271.9,489.1 L 271.0,490.0 L 271.8,491.8 L 284.0,491.5 L 286.8,491.8 L 286.9,491.3 L 286.7,491.2 L 288.3,490.4 L 290.1,488.6 L 289.4,487.1 L 289.2,486.2 L 290.0,485.5 L 290.0,485.0 L 289.5,484.4 L 288.8,484.5 L 287.9,484.3 L 287.7,483.8 L 286.9,483.7 L 286.1,483.2 L 284.6,483.0 L 282.1,481.9 L 280.8,482.0 L 280.5,481.2 L 280.6,481.0 L 279.4,480.6 L 279.8,479.8 L 278.2,479.5 L 278.8,478.9 L 274.7,475.0 L 274.1,474.2 L 272.7,473.1 L 273.1,472.8 L 270.8,471.4 L 268.7,470.7 L 268.3,470.4 L 268.4,470.2 L 268.0,470.1 L 268.1,469.8 L 267.6,469.3 L 266.6,469.0 L 266.5,468.8 L 265.2,468.6 L 264.8,468.3 L 265.0,468.1 L 264.5,467.9 L 264.9,467.6 L 264.9,467.2 L 263.8,466.5 L 262.6,466.0 L 258.2,467.0 L 258.8,467.4 L 258.2,467.8 L 257.6,467.7 L 257.7,468.6 L 257.1,469.2 L 255.9,469.2 L 252.4,470.5 L 252.5,470.1 L 252.0,468.8 L 246.9,466.2 L 246.5,465.5 L 244.7,465.0 L 244.0,464.6 L 244.6,463.2 L 241.5,463.3 L 240.1,464.1 L 237.7,463.5 L 237.4,463.9 L 235.0,463.5 L 235.0,416.5 L 233.1,416.3 L 223.9,414.0 L 220.3,414.1 L 217.4,414.6 L 216.6,414.9 L 213.3,414.5 L 210.1,413.8 L 204.1,413.2 L 203.1,412.8 L 201.1,412.6 L 200.5,412.3 L 199.4,412.5 L 197.9,412.3 L 197.4,412.6 L 197.6,412.7 L 196.1,412.4 L 191.4,411.9 L 188.8,412.0 L 188.3,412.2 L 186.0,412.3 L 183.4,412.7 L 182.3,412.5 L 182.4,412.1 L 179.2,410.6 L 175.0,410.1 L 170.5,410.3 L 167.8,409.9 L 165.0,409.0 L 161.2,408.6 L 160.4,408.2 L 158.0,407.8 L 157.1,407.9 L 155.3,408.4 L 152.3,410.0 L 150.3,410.5 L 148.3,410.6 L 146.3,410.2 L 143.8,410.4 L 141.2,410.8 L 137.9,412.2 L 135.5,412.9 L 133.4,413.3 L 130.3,413.1 L 127.7,413.9 L 124.3,415.8 L 123.6,416.8 L 123.6,417.8 L 123.1,418.4 L 119.9,419.8 L 113.2,420.5 L 109.0,420.3 L 108.3,420.5 L 108.3,421.5 L 107.8,422.6 L 105.4,423.1 L 105.1,423.4 L 108.2,423.8 L 110.2,424.6 L 112.5,424.9 L 119.1,427.3 L 120.6,428.9 L 120.0,429.1 L 120.4,429.5 L 125.8,430.4 L 121.9,431.9 L 117.7,431.9 L 111.1,433.0 L 102.4,435.6 L 99.6,436.2 L 98.9,436.8 L 99.2,437.3 L 102.0,438.1 L 105.3,438.5 L 104.6,439.0 L 104.6,439.3 L 107.4,440.6 L 107.1,440.9 L 107.0,441.3 L 107.5,441.9 L 108.6,442.3 L 108.2,442.5 L 108.6,442.8 L 109.3,442.8 L 109.6,442.4 L 114.8,443.1 L 118.2,442.5 L 120.9,442.3 L 122.3,442.5 L 123.8,443.2 L 126.0,443.6 L 127.2,443.2 L 127.6,442.6 L 128.4,442.3 L 132.0,443.3 L 132.9,443.2 L 134.6,444.0 L 134.8,444.9 L 135.6,446.0 L 133.6,447.4 L 130.8,447.5 L 130.2,447.2 L 128.9,447.0 L 128.5,446.6 L 127.1,446.5 L 125.9,447.1 L 126.6,447.5 L 127.8,447.5 L 126.5,448.5 L 125.6,448.7 L 124.4,449.5 L 123.7,449.5 L 121.7,448.8 L 119.7,448.4 L 117.5,448.7 L 116.3,449.5 L 115.6,449.7 L 115.4,450.5 L 114.8,450.8 L 115.1,451.2 L 113.8,451.4 L 113.9,451.9 L 113.3,452.1 L 112.8,453.0 L 110.5,454.4 L 109.5,454.6 L 109.4,455.6 L 108.9,456.0 L 108.6,457.2 L 109.0,457.7 L 109.8,458.0 L 110.0,458.4 L 111.1,459.0 L 111.7,459.7 L 112.9,459.9 L 113.8,459.8 L 113.5,460.2 L 114.1,460.6 L 114.1,461.0 L 114.4,461.3 L 112.7,461.9 L 112.4,462.2 L 112.5,462.4 L 113.8,462.8 L 115.1,463.6 L 116.3,463.8 L 117.0,464.2 L 117.6,464.7 L 118.3,465.0 L 117.8,465.2 L 117.9,465.6 L 119.9,466.8 L 120.9,466.9 L 121.3,466.7 L 120.0,466.2 L 125.3,466.2 L 126.5,465.5 L 127.8,465.6 L 128.2,465.9 L 129.1,465.9 L 130.6,467.4 L 129.7,468.0 L 129.1,468.8 L 129.5,469.4 L 130.5,470.1 L 130.7,471.0 L 130.3,471.4 L 128.7,471.6 L 128.6,471.8 L 128.9,472.0 L 130.6,472.1 L 130.9,472.5 L 131.5,472.5 L 133.8,471.8 L 134.3,472.5 L 135.3,472.5 L 136.0,472.2 L 137.0,471.0 L 137.0,470.8 L 138.3,470.6 L 138.6,471.1 L 139.6,471.0 L 140.3,471.4 L 141.5,471.0 L 142.3,471.1 L 144.6,473.3 L 146.1,473.2 L 147.0,472.7 L 148.9,472.3 L 149.7,472.6 L 145.5,480.3 L 138.9,482.8 L 137.8,483.5 L 136.9,484.7 L 133.9,484.7 L 130.9,485.3 L 128.6,486.4 L 127.1,487.4 L 125.3,488.1 L 124.3,488.8 L 122.2,489.5 L 120.4,489.5 L 118.5,490.2 L 117.4,490.2 L 116.9,490.4 L 116.1,491.5 L 115.0,491.9 L 114.8,492.1 L 115.0,492.7 L 116.0,493.3 L 117.9,493.2 L 119.4,492.2 L 122.2,492.3 L 122.8,492.0 L 124.9,491.9 L 125.2,491.7 L 125.2,491.5 L 124.3,490.9 L 124.7,490.6 L 125.8,490.8 L 126.8,490.3 L 127.1,490.5 L 127.0,490.9 L 127.6,491.0 L 127.2,491.3 L 127.7,491.7 L 127.6,492.3 L 128.2,492.5 L 128.8,491.8 L 129.8,491.8 L 129.7,491.5 L 130.8,491.6 L 132.0,491.2 L 132.1,490.9 L 131.4,490.5 L 131.6,490.2 L 133.1,490.1 L 133.2,489.9 L 132.9,489.7 L 133.6,489.5 L 133.8,489.0 L 134.9,488.7 L 134.5,488.4 L 133.8,488.5 L 135.5,487.9 L 135.7,487.9 L 135.2,488.4 L 135.5,489.5 L 137.0,489.5 L 137.4,489.9 L 137.8,489.7 L 137.8,489.5 L 138.2,489.0 L 138.6,489.0 L 138.9,488.5 L 138.8,488.3 L 139.0,488.3 L 139.4,488.8 L 138.6,489.4 L 138.8,489.9 L 138.3,490.3 L 138.7,491.0 L 140.2,490.4 L 140.9,490.5 L 141.4,489.9 L 142.2,490.0 L 142.3,490.4 L 143.2,490.9 L 143.9,490.9 L 144.4,490.4 L 143.8,490.0 L 144.0,489.8 L 143.8,489.5 L 142.9,489.4 L 142.9,488.4 L 142.3,488.4 L 141.3,488.9 L 141.0,488.4 L 139.7,488.0 L 139.8,487.8 L 140.2,487.5 L 142.3,487.3 L 142.4,487.0 L 142.8,486.8 L 150.0,486.8 L 154.5,483.8 L 154.4,483.7 L 158.3,481.3 L 158.4,479.5 L 158.9,479.2 L 158.8,478.8 L 159.2,478.3 L 160.3,477.9 L 160.8,477.5 L 161.7,477.5 L 161.8,477.0 L 163.5,476.8 L 164.2,476.2 L 164.2,476.0 L 164.8,475.9 L 165.3,475.1 L 168.0,475.0 L 169.3,474.5 L 170.2,473.4 L 170.7,473.1 L 170.6,472.7 L 170.9,472.4 L 171.9,472.5 L 172.7,472.1 L 172.6,471.8 L 174.1,471.4 L 174.0,471.0 L 174.2,470.7 L 174.1,470.5 L 174.9,470.0 L 182.8,469.5 L 185.0,469.1 L 191.9,467.4 L 192.7,466.3 L 193.6,465.8 L 195.1,465.5 L 198.9,465.6 L 200.2,466.3 L 201.6,466.3 L 203.3,465.8 L 203.8,465.0 L 204.5,464.6 L 210.1,463.4 L 217.0,464.6 L 217.3,464.9 L 216.1,465.9 L 216.1,466.1 L 216.6,466.4 L 217.9,466.3 L 219.3,465.4 L 226.2,464.7 L 230.0,465.1 L 235.4,466.5 L 237.9,466.7 L 248.7,470.1 L 249.9,470.7 L 250.0,471.2 L 251.4,472.1 L 254.4,473.4 L 254.9,473.3 L 255.9,474.1 L 256.7,474.3 L 257.4,474.1 L 256.8,474.5 L 256.6,474.9 L 256.8,475.6 L 257.7,475.8 Z M 1836.9,505.3 L 1837.3,505.7 L 1838.1,505.9 L 1838.9,505.7 L 1839.3,505.2 L 1838.3,504.6 L 1837.2,504.9 L 1836.9,505.3 Z M 1832.7,506.8 L 1832.8,507.0 L 1833.9,507.5 L 1834.5,507.5 L 1835.9,508.5 L 1837.6,508.5 L 1837.9,508.2 L 1837.1,507.7 L 1835.8,507.4 L 1835.2,506.8 L 1833.0,506.4 L 1832.7,506.8 Z M 1830.6,505.8 L 1831.2,506.4 L 1832.0,506.5 L 1832.5,506.3 L 1831.7,505.6 L 1831.0,505.6 L 1830.6,505.8 Z M 1830.0,504.8 L 1830.3,505.3 L 1831.0,505.3 L 1832.5,505.8 L 1833.3,505.3 L 1833.2,505.0 L 1832.6,504.8 L 1831.3,504.8 L 1830.8,504.5 L 1830.0,504.8 Z M 1829.7,505.6 L 1830.0,505.9 L 1830.5,505.7 L 1830.2,505.4 L 1829.7,505.6 Z M 1826.4,504.4 L 1826.7,504.6 L 1827.2,504.4 L 1826.9,504.1 L 1826.4,504.4 Z M 1825.6,505.5 L 1826.7,506.2 L 1827.1,506.0 L 1827.1,505.6 L 1829.1,505.5 L 1829.4,505.2 L 1828.5,505.0 L 1828.8,504.4 L 1827.9,504.1 L 1827.5,504.2 L 1827.1,504.8 L 1825.6,505.5 Z M 1818.8,503.1 L 1819.5,503.6 L 1820.0,503.5 L 1820.3,503.2 L 1819.4,502.8 L 1818.8,503.1 Z M 1808.9,501.1 L 1809.2,501.4 L 1810.1,501.6 L 1810.8,501.7 L 1811.2,501.5 L 1811.1,501.2 L 1810.6,501.0 L 1809.4,500.8 L 1808.9,501.1 Z M 1806.4,503.0 L 1806.6,503.2 L 1808.7,503.5 L 1809.1,503.2 L 1808.9,502.7 L 1809.3,502.4 L 1808.0,502.2 L 1806.7,502.6 L 1806.4,503.0 Z M 1801.7,500.4 L 1804.5,501.6 L 1805.2,501.3 L 1806.8,501.5 L 1807.1,501.4 L 1807.0,501.1 L 1807.9,500.9 L 1807.5,500.5 L 1807.0,500.5 L 1806.6,500.0 L 1805.7,499.8 L 1803.1,499.7 L 1801.7,500.4 Z M 99.3,495.2 L 99.7,495.6 L 100.2,495.4 L 100.0,495.1 L 99.3,495.2 Z M 90.6,481.9 L 91.0,482.3 L 92.1,482.6 L 93.1,482.1 L 92.5,481.7 L 91.0,481.6 L 90.6,481.9 Z M 89.7,479.1 L 90.2,479.4 L 90.8,479.1 L 90.3,478.8 L 89.7,479.1 Z M 88.6,501.3 L 88.8,501.6 L 89.1,501.7 L 90.8,501.2 L 91.7,501.4 L 92.1,501.0 L 91.8,500.4 L 92.1,499.8 L 91.5,499.4 L 90.9,499.4 L 90.6,499.7 L 90.9,500.3 L 89.4,500.2 L 89.0,500.4 L 89.2,500.8 L 88.6,501.3 Z M 87.4,479.0 L 87.5,479.4 L 87.9,479.5 L 87.5,479.6 L 87.5,479.8 L 88.0,480.0 L 89.4,479.3 L 89.9,478.7 L 89.2,478.5 L 87.8,478.7 L 87.4,479.0 Z M 85.4,502.2 L 86.2,502.6 L 87.3,502.1 L 87.6,501.6 L 87.4,501.4 L 86.6,501.2 L 85.7,501.7 L 85.4,502.2 Z M 83.0,502.6 L 83.2,502.9 L 84.0,503.0 L 84.9,502.0 L 84.1,501.8 L 83.1,502.3 L 83.0,502.6 Z M 76.4,503.7 L 76.6,503.9 L 77.4,504.0 L 78.9,503.4 L 78.6,503.0 L 77.8,502.8 L 76.8,503.1 L 76.4,503.7 Z M 63.9,503.8 L 64.2,504.2 L 64.8,503.9 L 64.4,503.6 L 63.9,503.8 Z M 61.9,504.1 L 62.4,504.5 L 63.0,504.1 L 62.5,503.8 L 61.9,504.1 Z M 52.2,505.4 L 52.7,505.8 L 53.3,505.5 L 52.7,505.1 L 52.2,505.4 Z M 48.5,505.6 L 49.3,506.2 L 49.0,506.4 L 49.1,506.8 L 50.5,507.3 L 51.1,506.8 L 51.7,507.0 L 53.0,506.6 L 53.6,506.9 L 54.4,506.7 L 55.1,505.8 L 55.2,506.3 L 54.6,506.9 L 55.3,507.3 L 58.9,506.3 L 60.2,506.2 L 60.9,506.0 L 61.0,505.7 L 63.3,505.2 L 64.7,505.3 L 65.7,505.1 L 67.6,505.2 L 68.6,504.8 L 69.7,504.7 L 70.7,505.0 L 72.4,505.1 L 73.8,505.0 L 74.2,505.3 L 74.7,504.9 L 75.4,504.9 L 75.9,504.5 L 75.5,504.2 L 73.4,504.2 L 72.5,503.9 L 70.0,504.1 L 70.5,503.5 L 70.4,503.1 L 69.2,502.6 L 68.4,502.7 L 67.5,503.2 L 67.3,503.4 L 67.5,503.7 L 66.0,504.2 L 65.4,504.2 L 62.3,504.9 L 61.3,504.9 L 60.0,504.2 L 59.3,504.1 L 58.6,504.5 L 58.7,505.4 L 57.7,505.5 L 57.6,504.9 L 57.0,504.7 L 55.9,505.0 L 55.1,505.7 L 54.9,505.2 L 54.0,505.0 L 53.7,505.2 L 53.5,505.8 L 52.3,506.1 L 52.2,505.5 L 50.9,505.5 L 50.3,505.1 L 48.9,505.2 L 48.5,505.6 Z M 47.8,507.6 L 48.5,507.9 L 49.1,507.6 L 48.6,507.3 L 47.8,507.6 Z M 45.2,506.1 L 46.0,506.5 L 46.5,506.4 L 46.8,506.0 L 45.8,505.6 L 45.2,506.1 Z M 44.2,507.1 L 44.5,507.4 L 45.3,507.2 L 46.0,507.6 L 47.2,507.5 L 47.9,507.1 L 46.8,506.7 L 45.3,507.1 L 44.6,506.8 L 44.2,507.1 Z M 43.9,508.6 L 44.1,509.1 L 44.6,509.1 L 45.8,508.4 L 45.9,508.2 L 45.0,507.7 L 43.9,508.6 Z M 102.2,463.9 L 103.0,464.9 L 106.7,466.0 L 108.4,466.1 L 109.1,466.5 L 109.9,466.5 L 111.3,465.8 L 112.5,465.7 L 113.0,465.2 L 112.2,464.7 L 112.1,463.4 L 111.3,463.0 L 110.1,463.1 L 109.2,462.4 L 108.3,462.9 L 107.5,462.8 L 106.2,463.1 L 105.3,463.6 L 103.4,463.5 L 102.2,463.9 Z M 75.6,464.0 L 76.0,464.3 L 76.8,464.0 L 76.3,463.7 L 75.6,464.0 Z M 73.8,461.7 L 74.3,462.1 L 74.2,462.7 L 75.3,463.1 L 75.6,463.4 L 78.5,463.8 L 79.5,463.5 L 78.3,462.9 L 76.2,462.4 L 75.9,461.9 L 75.3,461.7 L 74.9,461.3 L 74.2,461.4 L 73.8,461.7 Z M 140.5,490.9 L 140.8,491.3 L 141.6,491.2 L 142.3,491.5 L 142.9,491.1 L 142.2,490.6 L 141.4,490.5 L 140.5,490.9 Z M 134.0,489.2 L 134.5,489.5 L 135.0,489.2 L 134.5,489.0 L 134.0,489.2 Z M 125.9,490.9 L 126.3,491.2 L 126.8,491.0 L 126.4,490.7 L 125.9,490.9 Z M 124.9,493.1 L 126.5,493.8 L 127.3,493.8 L 128.8,493.2 L 128.2,492.7 L 125.7,492.2 L 125.3,492.4 L 124.9,493.1 Z M 123.6,487.8 L 123.8,488.1 L 124.2,488.3 L 124.9,487.9 L 124.1,487.3 L 123.7,487.5 L 123.6,487.8 Z M 110.2,495.4 L 111.5,494.8 L 112.2,495.2 L 113.0,494.9 L 113.6,495.1 L 115.4,494.9 L 116.5,494.1 L 116.6,493.8 L 116.0,493.6 L 114.7,494.1 L 113.4,494.3 L 113.8,493.9 L 113.3,493.7 L 112.8,493.3 L 111.7,493.3 L 111.2,493.8 L 110.3,493.6 L 109.3,493.9 L 109.0,494.4 L 109.3,494.8 L 106.2,494.7 L 104.7,495.0 L 103.9,495.6 L 103.8,495.9 L 104.3,496.5 L 103.8,496.9 L 103.9,497.2 L 101.2,497.9 L 101.5,497.4 L 101.2,497.2 L 99.6,496.9 L 98.6,497.1 L 97.6,497.7 L 97.6,498.1 L 95.9,498.9 L 95.6,499.2 L 95.6,499.6 L 95.2,499.7 L 95.0,500.1 L 93.8,500.3 L 93.7,500.6 L 94.1,500.7 L 93.4,501.2 L 94.1,501.4 L 96.3,500.7 L 97.2,500.2 L 98.0,500.4 L 98.4,500.1 L 98.3,499.6 L 99.8,498.8 L 102.0,499.1 L 104.5,498.2 L 106.5,498.0 L 108.1,497.0 L 110.1,496.1 L 110.4,495.8 L 110.2,495.4 Z M 139.5,472.0 L 140.0,472.3 L 140.7,472.1 L 140.5,471.8 L 140.0,471.7 L 139.5,472.0 Z M 137.3,471.6 L 137.6,471.9 L 138.1,471.9 L 138.0,472.2 L 138.5,472.3 L 139.5,471.5 L 139.3,471.3 L 137.9,471.0 L 137.3,471.6 Z M 177.7,470.4 L 178.0,470.8 L 178.4,470.9 L 179.3,470.7 L 179.9,470.9 L 180.7,470.4 L 180.3,470.1 L 179.3,469.9 L 177.9,470.0 L 177.7,470.4 Z M 165.3,478.5 L 166.9,479.2 L 167.0,480.1 L 168.1,480.5 L 168.1,480.9 L 168.9,481.4 L 168.8,481.7 L 168.2,481.8 L 167.3,481.7 L 166.2,482.2 L 165.5,482.8 L 165.7,483.1 L 166.4,483.2 L 167.5,482.7 L 170.4,482.7 L 171.0,482.5 L 171.2,482.1 L 170.0,481.8 L 171.1,481.5 L 171.2,481.1 L 172.6,480.7 L 172.8,480.1 L 174.0,480.2 L 174.5,479.8 L 175.8,479.5 L 176.1,479.2 L 175.8,478.9 L 176.8,478.8 L 177.9,478.1 L 178.3,478.4 L 179.1,478.2 L 179.2,478.0 L 178.9,477.6 L 179.7,477.1 L 179.8,476.7 L 179.3,475.9 L 178.7,475.7 L 178.8,475.2 L 177.6,474.8 L 180.2,474.2 L 181.1,474.4 L 181.5,474.1 L 181.5,473.7 L 181.2,473.5 L 181.5,473.2 L 179.1,472.7 L 179.6,472.6 L 179.7,472.4 L 179.1,472.1 L 178.8,471.6 L 178.1,471.6 L 177.8,471.2 L 177.3,471.3 L 172.7,474.5 L 172.5,474.7 L 172.6,474.9 L 170.4,475.4 L 168.8,476.4 L 167.1,476.9 L 165.6,478.1 L 165.3,478.5 Z M 160.8,486.0 L 161.0,486.4 L 162.4,486.5 L 162.6,486.3 L 162.7,485.6 L 161.8,485.4 L 160.8,486.0 Z M 155.5,484.2 L 155.9,484.5 L 155.8,484.9 L 156.0,485.2 L 156.8,485.2 L 157.3,484.8 L 157.2,484.5 L 156.6,484.2 L 156.7,484.0 L 156.5,483.7 L 155.8,483.6 L 155.5,484.2 Z M 207.5,468.0 L 208.2,468.3 L 209.3,467.8 L 208.7,467.2 L 208.2,467.1 L 207.9,467.2 L 207.9,467.7 L 207.5,468.0 Z M 132.8,444.3 L 132.9,444.6 L 133.5,444.7 L 133.9,444.6 L 134.1,444.3 L 133.4,444.1 L 132.8,444.3 Z M 130.7,446.9 L 131.2,447.2 L 131.8,447.0 L 131.2,446.7 L 130.7,446.9 Z M 98.9,440.1 L 99.5,440.4 L 100.4,440.1 L 100.2,439.9 L 99.6,439.8 L 98.9,440.1 Z M 95.7,436.9 L 96.2,437.1 L 96.9,436.9 L 96.3,436.6 L 95.7,436.9 Z M 95.1,436.4 L 95.6,436.6 L 96.2,436.2 L 95.8,435.9 L 95.2,435.9 L 95.1,436.4 Z M 80.2,447.4 L 80.2,447.6 L 80.8,448.3 L 82.0,448.6 L 83.0,448.7 L 84.8,448.1 L 87.0,448.4 L 87.8,448.7 L 88.2,449.3 L 89.4,449.4 L 90.5,449.8 L 90.8,450.4 L 91.8,450.6 L 92.8,450.3 L 92.7,449.8 L 93.7,449.4 L 94.9,449.4 L 95.4,449.5 L 95.4,449.8 L 96.2,449.9 L 96.5,449.7 L 96.3,449.3 L 97.1,448.5 L 91.0,447.5 L 90.4,447.4 L 90.1,446.9 L 88.5,446.2 L 86.7,446.4 L 85.2,446.9 L 82.3,446.4 L 82.3,446.0 L 81.6,445.8 L 80.8,446.0 L 80.7,446.6 L 80.2,447.4 Z" fill="#e5e7eb" stroke="#fff" stroke-width="0.7"><title>Alaska: no records</title></path><path d="M 619.0,242.3 L 618.7,241.5 L 618.8,241.0 L 619.9,240.4 L 621.0,238.9 L 620.7,238.5 L 621.0,238.2 L 620.1,237.0 L 620.6,235.6 L 620.2,235.0 L 620.5,234.7 L 610.0,237.4 L 610.1,238.7 L 610.5,239.1 L 610.5,240.6 L 611.4,241.9 L 611.3,242.5 L 611.6,243.3 L 611.3,244.0 L 611.3,245.2 L 612.3,247.3 L 612.2,248.9 L 612.5,249.0 L 612.5,248.5 L 612.9,248.4 L 613.4,249.0 L 614.8,255.4 L 619.6,254.4 L 618.7,253.5 L 618.7,252.7 L 619.1,252.1 L 618.3,247.9 L 619.2,243.9 L 619.0,243.8 L 619.3,243.0 L 619.0,242.3 Z" fill="#e5e7eb" stroke="#fff" stroke-width="0.7"><title>Vermont: no records</title></path><path d="M 622.3,266.9 L 626.1,264.5 L 624.8,259.0 L 619.2,260.3 L 618.9,260.7 L 618.9,260.4 L 614.9,261.3 L 616.0,267.4 L 616.5,268.0 L 615.3,269.3 L 616.0,270.1 L 616.3,270.3 L 622.3,266.9 Z" fill="#fff7ec" stroke="#fff" stroke-width="0.7"><title>Connecticut: 5 (explicit campuses 0; building-inferred sites 4; standalone points 1)</title></path><path d="M 607.2,283.1 L 607.6,282.1 L 606.2,282.1 L 605.4,283.2 L 608.2,293.3 L 612.6,292.4 L 611.8,289.7 L 609.1,286.4 L 607.3,285.1 L 607.4,284.1 L 607.0,283.7 L 607.2,283.1 Z" fill="#e5e7eb" stroke="#fff" stroke-width="0.7"><title>Delaware: no records</title></path><path d="M 431.3,355.9 L 434.0,321.6 L 434.3,321.6 L 434.6,317.5 L 395.6,313.4 L 389.2,359.7 L 395.0,360.5 L 395.5,356.8 L 407.1,358.2 L 406.4,357.5 L 406.6,356.4 L 431.1,358.7 L 431.3,355.9 Z" fill="#fee8c8" stroke="#fff" stroke-width="0.7"><title>New Mexico: 11 (explicit campuses 1; building-inferred sites 5; standalone points 5)</title></path><path d="M 597.4,311.7 L 573.1,315.5 L 572.9,317.0 L 573.2,317.6 L 572.3,317.8 L 571.4,319.7 L 570.8,319.9 L 570.1,319.6 L 569.3,320.0 L 568.1,321.6 L 567.8,321.5 L 567.8,320.9 L 567.5,320.7 L 566.7,321.4 L 566.6,322.0 L 565.8,322.1 L 566.0,322.5 L 565.5,323.4 L 564.4,323.7 L 562.4,325.6 L 560.8,325.8 L 560.0,326.3 L 559.2,327.3 L 559.1,328.5 L 558.0,328.6 L 557.6,329.1 L 557.6,331.0 L 565.6,329.9 L 567.9,329.0 L 570.1,327.5 L 570.3,327.7 L 579.1,326.6 L 579.2,327.5 L 579.8,326.9 L 581.0,328.1 L 581.1,329.1 L 588.5,328.0 L 597.7,334.6 L 600.0,333.8 L 601.5,334.4 L 601.8,334.1 L 602.0,331.2 L 603.4,328.8 L 605.4,326.8 L 607.0,325.9 L 608.3,325.6 L 609.7,326.2 L 610.6,324.0 L 612.1,321.8 L 613.3,320.5 L 615.3,319.3 L 614.9,315.6 L 612.0,311.3 L 610.9,308.9 L 597.4,311.7 Z" fill="#fdbb84" stroke="#fff" stroke-width="0.7"><title>North Carolina: 17 (explicit campuses 5; building-inferred sites 8; standalone points 4)</title></path><path d="M 516.6,241.7 L 513.4,241.0 L 512.7,239.7 L 511.6,239.3 L 513.8,233.2 L 509.9,233.3 L 502.5,238.6 L 502.1,238.2 L 501.5,238.5 L 501.4,239.1 L 500.9,239.0 L 501.1,243.8 L 500.8,244.3 L 500.4,244.3 L 498.8,245.4 L 498.4,246.5 L 497.9,246.8 L 497.9,248.0 L 498.7,248.2 L 499.3,249.1 L 498.7,250.3 L 498.8,251.8 L 498.5,252.2 L 498.8,253.2 L 498.6,254.8 L 500.1,256.2 L 501.4,256.3 L 502.0,257.2 L 503.6,257.8 L 504.2,259.1 L 507.3,261.1 L 508.0,262.4 L 507.9,263.3 L 508.4,265.8 L 509.4,266.6 L 508.7,267.7 L 509.0,269.5 L 509.6,270.9 L 511.7,271.6 L 512.2,272.7 L 533.9,271.1 L 532.4,263.9 L 532.7,257.7 L 534.0,251.4 L 534.9,249.6 L 536.2,248.2 L 533.1,246.8 L 531.1,247.0 L 529.6,249.1 L 529.4,250.2 L 528.2,250.0 L 527.6,249.3 L 528.0,247.9 L 526.7,248.1 L 527.1,246.9 L 526.8,246.4 L 527.2,246.2 L 526.8,245.7 L 527.1,245.4 L 526.5,244.9 L 525.0,244.5 L 525.3,244.0 L 525.1,243.6 L 522.7,243.0 L 521.8,243.3 L 519.3,242.3 L 516.6,241.7 Z" fill="#fee8c8" stroke="#fff" stroke-width="0.7"><title>Wisconsin: 13 (explicit campuses 1; building-inferred sites 9; standalone points 3)</title></path><path d="M 335.9,227.9 L 333.9,227.3 L 331.0,227.9 L 330.1,227.4 L 328.5,226.2 L 328.9,223.7 L 328.7,222.6 L 327.8,221.5 L 326.3,221.3 L 325.9,220.2 L 323.8,219.8 L 323.1,219.1 L 323.0,219.3 L 322.3,219.2 L 322.6,220.1 L 322.6,221.2 L 321.9,221.8 L 321.9,222.6 L 321.6,223.3 L 321.7,224.0 L 320.5,226.5 L 320.3,227.9 L 319.0,230.4 L 316.4,237.2 L 314.7,240.6 L 313.4,241.6 L 313.2,242.7 L 312.3,244.1 L 310.8,245.6 L 311.7,247.3 L 311.6,247.8 L 310.8,248.4 L 311.1,249.0 L 310.7,250.3 L 310.9,252.1 L 314.0,253.3 L 353.8,263.7 L 357.0,249.0 L 357.9,247.8 L 357.8,247.0 L 358.4,246.5 L 358.1,245.8 L 356.9,245.2 L 356.9,244.2 L 358.5,241.8 L 359.3,241.5 L 360.0,240.7 L 360.2,239.7 L 361.1,238.9 L 361.8,237.5 L 363.4,235.5 L 363.2,234.2 L 362.0,233.3 L 361.6,231.8 L 350.2,229.1 L 349.2,229.4 L 346.5,228.8 L 346.0,229.2 L 344.3,229.1 L 342.8,229.5 L 341.2,229.4 L 340.5,228.7 L 338.8,229.2 L 337.9,228.8 L 337.3,229.1 L 337.1,228.5 L 335.9,227.9 Z" fill="#fc8d59" stroke="#fff" stroke-width="0.7"><title>Oregon: 50 (explicit campuses 6; building-inferred sites 42; standalone points 2)</title></path><path d="M 480.4,282.8 L 480.5,282.3 L 479.9,282.1 L 480.0,281.6 L 479.8,281.8 L 479.4,281.6 L 479.3,280.4 L 479.6,280.2 L 479.4,279.9 L 479.6,279.4 L 479.0,278.6 L 479.2,278.0 L 478.5,277.8 L 478.4,276.8 L 477.9,276.4 L 478.0,275.6 L 477.5,274.8 L 477.7,273.8 L 476.3,273.6 L 475.7,272.7 L 475.9,272.3 L 473.2,271.3 L 472.2,270.5 L 469.0,270.5 L 467.9,271.3 L 465.4,269.9 L 465.1,269.3 L 432.0,267.5 L 430.6,283.9 L 442.9,284.8 L 442.3,293.1 L 484.3,294.3 L 483.6,294.0 L 483.8,293.3 L 483.2,292.7 L 483.2,292.3 L 482.1,291.7 L 482.3,291.5 L 481.9,290.1 L 482.1,289.8 L 481.5,289.9 L 481.5,289.3 L 480.7,288.3 L 481.2,286.9 L 480.7,285.5 L 480.7,284.7 L 481.0,284.6 L 480.4,284.4 L 480.4,283.6 L 480.8,283.5 L 480.3,283.2 L 480.4,282.8 Z" fill="#fee8c8" stroke="#fff" stroke-width="0.7"><title>Nebraska: 14 (explicit campuses 5; building-inferred sites 9; standalone points 0)</title></path><path d="M 583.4,356.3 L 583.4,354.7 L 584.9,352.1 L 582.5,351.5 L 582.0,350.5 L 582.1,349.7 L 581.5,349.1 L 581.5,348.7 L 579.8,347.6 L 579.5,346.1 L 578.8,345.2 L 578.8,344.6 L 576.9,343.7 L 576.7,343.2 L 576.1,343.0 L 576.0,342.5 L 575.4,342.3 L 575.3,341.3 L 573.4,340.3 L 572.2,338.8 L 570.5,338.0 L 569.2,336.5 L 567.9,334.1 L 566.7,334.0 L 565.7,333.2 L 564.4,332.7 L 564.5,331.6 L 565.7,330.4 L 565.7,329.9 L 549.1,332.0 L 553.9,349.2 L 555.8,352.6 L 555.6,353.4 L 556.5,353.9 L 555.4,355.0 L 555.6,355.9 L 555.2,357.1 L 555.2,358.0 L 556.1,359.9 L 556.0,362.8 L 558.2,366.5 L 576.8,365.2 L 577.2,366.7 L 578.1,366.8 L 578.3,365.1 L 577.8,363.8 L 578.3,362.8 L 580.8,363.4 L 582.7,363.2 L 582.2,362.8 L 582.6,361.0 L 582.2,360.5 L 583.1,358.6 L 582.9,357.4 L 583.4,356.3 Z" fill="#fdbb84" stroke="#fff" stroke-width="0.7"><title>Georgia: 30 (explicit campuses 7; building-inferred sites 20; standalone points 3)</title></path><path d="M 555.3,358.1 L 555.2,357.1 L 555.6,355.9 L 555.4,355.0 L 556.5,353.9 L 555.6,353.4 L 555.8,352.6 L 553.9,349.2 L 549.1,332.0 L 531.9,333.5 L 532.7,334.4 L 532.1,359.3 L 533.9,373.5 L 535.4,373.0 L 536.3,373.4 L 539.9,372.4 L 539.9,371.9 L 540.4,371.7 L 540.0,371.6 L 540.9,370.6 L 540.2,370.0 L 540.4,368.7 L 538.7,367.2 L 538.8,366.1 L 556.9,364.2 L 556.0,362.8 L 556.1,359.9 L 555.3,358.1 Z" fill="#fee8c8" stroke="#fff" stroke-width="0.7"><title>Alabama: 6 (explicit campuses 3; building-inferred sites 3; standalone points 0)</title></path><path d="M 386.5,270.1 L 371.4,267.5 L 363.6,308.1 L 395.6,313.4 L 400.1,280.5 L 388.0,278.7 L 389.3,270.5 L 386.5,270.1 Z" fill="#fee8c8" stroke="#fff" stroke-width="0.7"><title>Utah: 9 (explicit campuses 1; building-inferred sites 7; standalone points 1)</title></path><path d="M 574.8,279.1 L 572.8,267.5 L 568.6,269.1 L 562.4,274.5 L 560.7,274.8 L 558.2,273.6 L 557.8,272.8 L 556.2,274.9 L 547.9,276.3 L 550.3,297.6 L 550.8,297.2 L 551.7,297.7 L 552.6,297.2 L 552.8,297.7 L 553.6,298.0 L 554.4,299.6 L 556.6,299.7 L 557.4,300.4 L 558.0,300.6 L 558.9,299.8 L 560.4,300.5 L 561.3,300.2 L 561.9,299.2 L 562.8,298.9 L 563.3,300.2 L 564.2,300.4 L 565.1,301.4 L 566.7,300.9 L 566.8,299.8 L 567.5,299.5 L 566.9,298.0 L 567.8,295.9 L 568.5,296.1 L 568.8,297.0 L 569.2,296.4 L 569.7,296.5 L 569.1,295.2 L 569.5,295.0 L 569.4,294.4 L 569.7,293.6 L 570.4,293.5 L 570.4,292.9 L 571.0,292.2 L 571.5,292.7 L 572.5,292.2 L 574.3,289.9 L 574.4,289.1 L 574.1,288.7 L 574.5,287.8 L 574.3,287.4 L 574.6,287.4 L 574.5,286.0 L 575.1,283.9 L 574.8,283.4 L 574.9,282.7 L 574.3,281.8 L 575.1,281.2 L 574.8,279.1 Z" fill="#fc8d59" stroke="#fff" stroke-width="0.7"><title>Ohio: 45 (explicit campuses 9; building-inferred sites 35; standalone points 1)</title></path><path d="M 481.4,319.1 L 434.6,317.5 L 434.3,321.6 L 453.9,322.7 L 453.2,338.8 L 453.7,338.7 L 455.2,340.4 L 455.8,340.4 L 456.0,340.1 L 457.1,340.5 L 457.4,339.8 L 458.5,340.8 L 458.5,341.8 L 459.9,341.8 L 461.4,342.7 L 462.5,342.4 L 463.2,343.2 L 464.1,342.5 L 465.4,342.9 L 465.8,342.5 L 465.9,343.8 L 466.9,343.9 L 466.7,344.8 L 467.6,345.0 L 468.7,343.9 L 469.3,344.3 L 469.3,344.7 L 470.1,344.7 L 470.3,345.4 L 471.8,344.6 L 472.1,345.2 L 471.9,345.9 L 472.3,346.3 L 472.9,345.4 L 472.6,345.1 L 473.1,345.2 L 473.4,344.3 L 473.8,344.3 L 474.1,345.1 L 474.6,345.0 L 474.8,345.4 L 475.3,345.3 L 475.5,344.6 L 476.0,344.8 L 475.7,345.2 L 477.1,345.8 L 477.6,346.6 L 478.0,345.9 L 478.7,346.0 L 479.0,345.3 L 480.0,345.0 L 481.1,345.3 L 482.7,344.4 L 483.1,344.9 L 484.8,345.0 L 485.3,344.3 L 487.0,345.1 L 487.7,346.0 L 488.4,345.9 L 488.2,346.1 L 488.6,346.5 L 489.2,346.4 L 489.0,346.7 L 489.6,346.5 L 489.6,346.8 L 489.9,346.8 L 489.7,347.0 L 490.2,347.0 L 490.4,332.4 L 489.0,323.2 L 489.0,319.1 L 481.4,319.1 Z" fill="#fee8c8" stroke="#fff" stroke-width="0.7"><title>Oklahoma: 6 (explicit campuses 0; building-inferred sites 4; standalone points 2)</title></path><path d="M 558.2,317.6 L 541.9,318.9 L 533.1,319.9 L 533.1,319.6 L 531.6,319.7 L 531.9,321.1 L 523.7,321.7 L 522.6,322.2 L 522.2,321.9 L 522.4,323.0 L 521.7,323.3 L 522.3,323.9 L 521.2,324.1 L 522.0,324.8 L 521.2,326.1 L 521.8,327.0 L 521.1,326.8 L 521.0,327.2 L 521.5,327.5 L 519.8,328.4 L 520.1,329.0 L 520.6,328.9 L 519.9,329.6 L 520.3,330.1 L 519.5,329.9 L 519.7,330.9 L 519.4,331.2 L 519.0,330.5 L 518.6,331.3 L 519.2,331.3 L 518.7,332.2 L 519.2,332.5 L 519.1,332.9 L 519.4,333.3 L 518.8,333.5 L 518.6,334.3 L 518.0,334.2 L 517.9,334.6 L 557.6,331.0 L 557.6,329.1 L 558.0,328.6 L 559.1,328.5 L 559.2,327.3 L 560.0,326.3 L 560.8,325.8 L 562.4,325.6 L 564.4,323.7 L 565.5,323.4 L 566.0,322.5 L 565.8,322.1 L 566.6,322.0 L 566.7,321.4 L 567.5,320.7 L 567.8,320.9 L 567.8,321.5 L 568.1,321.6 L 569.3,320.0 L 570.1,319.6 L 570.8,319.9 L 571.4,319.7 L 572.3,317.8 L 573.2,317.6 L 572.9,317.0 L 573.3,315.3 L 558.2,317.6 Z" fill="#fee8c8" stroke="#fff" stroke-width="0.7"><title>Tennessee: 15 (explicit campuses 5; building-inferred sites 8; standalone points 2)</title></path><path d="M 416.8,282.6 L 430.6,283.9 L 433.4,251.1 L 393.2,246.1 L 388.0,278.7 L 416.8,282.6 Z" fill="#fee8c8" stroke="#fff" stroke-width="0.7"><title>Wyoming: 6 (explicit campuses 4; building-inferred sites 2; standalone points 0)</title></path><path d="M 626.5,252.5 L 626.4,252.9 L 614.8,255.4 L 614.7,261.0 L 614.8,261.3 L 618.9,260.4 L 618.9,260.7 L 619.2,260.3 L 627.3,258.4 L 628.0,260.1 L 629.5,260.9 L 630.3,262.6 L 631.0,263.1 L 631.5,262.8 L 632.1,264.0 L 632.6,264.0 L 632.9,263.2 L 633.6,262.9 L 636.6,263.0 L 637.7,262.4 L 637.6,261.8 L 636.6,260.9 L 636.1,261.0 L 636.3,261.7 L 636.1,261.8 L 635.2,261.6 L 634.6,262.0 L 634.3,261.4 L 633.5,261.2 L 635.8,259.6 L 635.8,260.5 L 636.5,260.6 L 636.9,260.0 L 636.8,258.2 L 636.0,256.6 L 635.0,255.7 L 634.1,255.6 L 631.6,256.2 L 631.3,256.0 L 630.8,255.3 L 630.7,253.1 L 631.1,252.2 L 630.8,251.6 L 629.7,251.7 L 629.2,250.6 L 628.2,250.7 L 627.6,251.2 L 627.5,251.6 L 626.9,251.8 L 626.9,252.4 L 626.5,252.5 Z" fill="#fff7ec" stroke="#fff" stroke-width="0.7"><title>Massachusetts: 3 (explicit campuses 0; building-inferred sites 3; standalone points 0)</title></path><path d="M 603.7,310.4 L 610.9,308.9 L 609.4,305.9 L 609.4,304.9 L 610.5,302.3 L 610.9,298.6 L 611.7,297.8 L 612.2,296.1 L 609.4,297.0 L 609.4,297.4 L 608.5,297.9 L 606.8,297.9 L 605.8,298.7 L 603.8,297.9 L 603.0,297.0 L 600.8,297.0 L 599.9,295.5 L 598.5,296.3 L 598.1,295.1 L 598.3,294.2 L 599.4,293.0 L 599.1,291.7 L 598.3,291.1 L 597.7,291.1 L 597.6,290.7 L 595.8,290.3 L 596.0,289.4 L 595.2,288.8 L 594.1,289.0 L 593.8,290.6 L 590.1,288.5 L 590.3,289.4 L 589.9,290.7 L 590.2,291.0 L 589.3,292.8 L 588.3,293.7 L 587.9,294.8 L 587.0,294.3 L 585.9,298.1 L 584.6,298.0 L 584.1,297.3 L 583.3,297.1 L 583.2,299.0 L 582.7,299.5 L 582.9,299.8 L 582.2,300.7 L 581.9,302.3 L 581.1,303.5 L 580.5,305.1 L 581.1,305.5 L 580.5,306.2 L 580.8,306.4 L 580.7,306.6 L 579.7,307.5 L 579.2,307.1 L 577.9,308.2 L 577.3,307.8 L 577.4,308.5 L 577.2,308.8 L 575.2,309.8 L 574.2,309.1 L 573.1,310.3 L 572.3,310.5 L 570.5,309.4 L 570.4,308.8 L 570.0,308.6 L 570.4,308.2 L 570.1,308.0 L 568.0,310.6 L 565.7,312.1 L 565.8,312.8 L 565.0,313.4 L 565.0,314.2 L 563.8,314.6 L 563.5,315.6 L 560.2,317.2 L 573.3,315.3 L 573.1,315.5 L 583.2,314.3 L 603.7,310.4 Z" fill="#b30000" stroke="#fff" stroke-width="0.7"><title>Virginia: 166 (explicit campuses 31; building-inferred sites 132; standalone points 3)</title></path><path d="M 480.8,265.4 L 476.5,265.4 L 476.5,265.9 L 476.9,266.3 L 476.9,267.0 L 476.5,267.3 L 476.7,267.6 L 477.2,267.7 L 477.4,268.5 L 476.9,269.2 L 477.0,269.8 L 476.2,271.7 L 476.9,272.5 L 477.1,273.7 L 477.7,273.8 L 477.5,274.8 L 478.0,275.6 L 477.9,276.3 L 478.4,276.8 L 478.3,277.4 L 479.2,278.0 L 479.0,278.6 L 479.6,279.5 L 479.4,279.9 L 479.6,280.2 L 479.3,280.4 L 479.4,281.5 L 479.8,281.8 L 480.0,281.5 L 479.9,282.1 L 480.5,282.3 L 480.3,283.2 L 480.8,283.6 L 480.4,283.6 L 480.4,284.4 L 481.0,284.6 L 480.7,284.7 L 480.7,285.5 L 481.2,286.9 L 480.7,288.2 L 481.4,288.9 L 481.4,289.5 L 506.4,288.6 L 508.0,290.5 L 508.5,290.4 L 508.7,289.5 L 508.4,289.0 L 510.1,288.0 L 510.2,286.7 L 511.0,285.6 L 511.0,284.4 L 509.9,283.3 L 510.2,281.8 L 513.7,280.7 L 514.4,280.1 L 514.5,278.8 L 515.3,278.3 L 515.4,276.6 L 515.2,275.7 L 513.8,274.9 L 513.4,273.8 L 512.1,272.9 L 512.2,272.6 L 511.7,271.6 L 509.5,270.8 L 508.7,268.2 L 508.7,267.7 L 509.4,266.6 L 508.4,265.9 L 508.3,264.7 L 480.8,265.4 Z" fill="#fdbb84" stroke="#fff" stroke-width="0.7"><title>Iowa: 17 (explicit campuses 5; building-inferred sites 12; standalone points 0)</title></path><path d="M 384.7,311.7 L 363.6,308.1 L 362.4,314.6 L 361.5,315.8 L 360.8,315.8 L 360.0,314.5 L 358.5,314.3 L 357.6,314.6 L 357.5,315.5 L 357.9,316.5 L 357.4,316.9 L 357.5,318.6 L 357.2,319.5 L 357.4,322.2 L 357.2,322.6 L 356.7,322.7 L 357.0,323.0 L 356.4,324.6 L 357.3,326.1 L 357.4,328.3 L 358.6,329.4 L 358.8,330.2 L 356.6,331.1 L 355.6,332.3 L 355.5,334.2 L 355.2,334.4 L 355.0,335.4 L 354.0,336.4 L 353.5,336.4 L 353.6,336.8 L 353.3,337.2 L 353.6,337.6 L 353.1,339.0 L 353.3,339.5 L 354.3,339.8 L 354.3,341.3 L 353.7,341.9 L 352.5,341.8 L 351.6,342.6 L 351.4,343.6 L 375.2,357.7 L 389.2,359.7 L 395.6,313.4 L 384.7,311.7 Z" fill="#fc8d59" stroke="#fff" stroke-width="0.7"><title>Arizona: 53 (explicit campuses 8; building-inferred sites 30; standalone points 15)</title></path><path d="M 463.7,343.1 L 463.2,343.2 L 462.5,342.4 L 461.4,342.7 L 459.9,341.8 L 458.5,341.8 L 458.5,340.8 L 457.5,339.8 L 457.1,340.5 L 456.0,340.1 L 455.8,340.4 L 455.2,340.4 L 453.7,338.7 L 453.2,338.8 L 453.9,322.7 L 434.0,321.6 L 431.1,358.7 L 406.6,356.4 L 406.4,357.2 L 407.3,358.5 L 408.0,358.8 L 409.0,361.1 L 410.7,362.1 L 411.8,363.8 L 412.9,364.6 L 414.1,366.7 L 415.4,367.2 L 417.2,369.0 L 417.5,370.8 L 418.5,372.2 L 418.4,374.7 L 419.4,377.2 L 421.4,378.7 L 421.7,379.4 L 422.6,380.1 L 424.3,380.7 L 424.7,381.4 L 425.9,381.7 L 427.7,383.3 L 428.9,383.4 L 429.9,381.8 L 430.9,381.5 L 430.6,381.2 L 430.8,380.5 L 431.3,380.1 L 431.4,379.1 L 432.5,377.4 L 433.9,377.1 L 434.6,377.4 L 435.1,376.4 L 436.8,377.3 L 438.6,377.2 L 439.8,377.8 L 440.6,377.3 L 440.6,377.8 L 441.6,377.7 L 441.8,378.7 L 442.2,378.8 L 442.2,379.3 L 442.6,379.0 L 442.5,379.8 L 443.9,380.4 L 444.2,381.2 L 445.5,382.1 L 445.8,382.9 L 446.5,383.5 L 446.6,385.0 L 447.4,386.0 L 447.6,387.1 L 448.7,388.5 L 448.4,388.7 L 448.9,390.4 L 450.4,391.5 L 451.0,392.8 L 451.4,392.9 L 451.8,394.4 L 454.3,396.4 L 454.2,396.9 L 454.6,397.1 L 454.1,398.4 L 454.8,399.0 L 454.7,400.8 L 455.9,402.3 L 457.2,406.0 L 459.2,406.3 L 460.2,407.4 L 461.9,407.6 L 463.7,409.0 L 467.6,409.2 L 468.7,410.5 L 469.8,410.8 L 469.8,410.3 L 470.4,409.9 L 471.9,409.9 L 471.7,407.6 L 470.4,402.8 L 470.3,400.5 L 471.2,397.3 L 473.0,394.3 L 474.6,392.6 L 477.3,390.9 L 478.3,389.9 L 484.8,386.5 L 486.6,384.5 L 489.7,382.7 L 489.8,381.8 L 494.0,379.9 L 495.6,380.1 L 494.7,378.3 L 496.3,376.3 L 496.1,376.0 L 496.4,375.6 L 496.2,374.4 L 495.8,374.0 L 496.2,373.1 L 495.9,372.3 L 497.2,369.9 L 497.0,369.4 L 497.4,369.0 L 497.0,368.5 L 497.5,368.2 L 497.1,367.7 L 497.3,367.0 L 496.9,367.1 L 496.6,366.3 L 496.2,366.0 L 496.5,365.4 L 495.7,364.7 L 495.9,364.3 L 495.1,363.7 L 495.3,362.7 L 495.1,362.2 L 494.5,361.2 L 493.6,360.4 L 493.1,347.3 L 491.0,347.6 L 491.0,347.3 L 490.4,347.2 L 490.5,346.8 L 490.0,347.1 L 489.6,346.5 L 488.6,346.5 L 488.2,346.1 L 488.4,345.9 L 487.7,346.1 L 487.0,345.1 L 485.3,344.3 L 484.8,345.0 L 483.1,344.9 L 482.7,344.4 L 481.1,345.3 L 480.0,345.0 L 479.0,345.3 L 478.7,346.0 L 478.0,345.9 L 477.6,346.6 L 477.1,345.8 L 475.8,345.2 L 476.0,344.9 L 475.5,344.6 L 475.3,345.3 L 474.8,345.4 L 474.6,345.0 L 474.1,345.1 L 473.8,344.3 L 473.3,344.3 L 473.3,344.9 L 472.6,345.1 L 472.9,345.4 L 472.4,346.3 L 471.9,345.9 L 472.1,345.2 L 471.8,344.6 L 470.3,345.4 L 470.1,344.7 L 469.3,344.7 L 469.3,344.3 L 468.7,343.9 L 467.3,345.1 L 466.6,344.7 L 466.9,343.9 L 465.9,343.8 L 465.8,342.5 L 465.4,342.9 L 464.1,342.5 L 463.7,343.1 Z" fill="#e34a33" stroke="#fff" stroke-width="0.7"><title>Texas: 93 (explicit campuses 10; building-inferred sites 81; standalone points 2)</title></path><path d="M 371.6,523.5 L 371.4,522.2 L 369.0,520.2 L 362.5,517.3 L 361.0,517.4 L 360.5,519.0 L 361.4,520.8 L 358.8,523.2 L 358.6,524.3 L 360.7,528.9 L 360.3,531.6 L 360.9,532.9 L 364.2,534.5 L 367.3,531.3 L 368.8,530.4 L 372.0,529.6 L 374.3,528.0 L 374.9,526.6 L 371.6,523.5 Z M 90.7,420.0 L 91.5,421.0 L 93.2,420.5 L 93.1,419.5 L 92.2,418.8 L 91.2,419.0 L 90.7,420.0 Z M 288.1,484.2 L 288.8,484.9 L 289.7,484.3 L 289.0,483.6 L 288.1,484.2 Z M 254.8,478.0 L 255.6,478.8 L 256.3,478.1 L 255.7,477.5 L 254.8,478.0 Z M 235.4,474.8 L 236.2,475.4 L 236.4,476.3 L 237.3,476.3 L 237.5,477.8 L 238.6,477.7 L 238.7,476.1 L 237.8,474.2 L 236.3,473.9 L 235.4,474.8 Z M 215.3,461.0 L 216.0,461.6 L 216.7,461.1 L 216.0,460.4 L 215.3,461.0 Z M 183.8,454.9 L 184.4,455.5 L 185.1,454.9 L 184.4,454.3 L 183.8,454.9 Z M 170.3,451.6 L 171.2,452.5 L 172.0,451.6 L 171.0,450.9 L 170.3,451.6 Z M 143.2,448.3 L 144.0,449.1 L 145.2,448.3 L 144.9,447.6 L 143.9,447.4 L 143.2,448.3 Z M 119.5,427.4 L 120.4,428.7 L 123.0,428.2 L 124.0,425.7 L 122.3,424.9 L 120.2,426.0 L 120.1,426.7 L 120.9,427.2 L 120.0,427.0 L 119.5,427.4 Z M 331.9,502.0 L 334.4,506.1 L 337.2,506.0 L 338.3,506.6 L 339.9,506.4 L 340.8,505.6 L 340.2,503.2 L 338.9,502.7 L 336.7,499.9 L 331.9,502.0 Z M 349.5,506.2 L 348.3,505.7 L 347.6,506.1 L 344.7,505.7 L 343.6,507.8 L 344.1,508.4 L 347.1,508.3 L 349.5,509.1 L 352.1,507.6 L 351.8,506.5 L 349.5,506.2 Z M 346.6,510.2 L 348.3,512.8 L 350.2,512.4 L 351.0,511.3 L 349.1,509.3 L 347.4,509.3 L 346.6,510.2 Z M 354.5,509.3 L 352.9,507.9 L 351.2,508.9 L 351.3,510.8 L 354.0,512.7 L 351.2,514.0 L 351.0,515.6 L 353.6,515.4 L 354.2,514.1 L 355.5,514.7 L 358.4,514.2 L 360.6,512.9 L 360.5,510.5 L 359.6,510.7 L 356.8,509.0 L 354.5,509.3 Z M 308.4,498.8 L 308.8,500.0 L 309.9,500.2 L 311.6,498.6 L 312.0,497.1 L 310.5,496.0 L 308.4,498.8 Z M 304.8,501.1 L 305.4,501.8 L 306.2,501.3 L 305.6,500.5 L 304.8,501.1 Z M 320.7,496.6 L 321.0,494.9 L 319.2,493.5 L 316.8,493.7 L 315.0,494.6 L 313.8,496.7 L 314.5,497.7 L 316.4,498.8 L 318.6,499.2 L 320.3,498.1 L 320.7,496.6 Z" fill="#e5e7eb" stroke="#fff" stroke-width="0.7"><title>Hawaii: no records</title></path><path d="M 463.7,220.1 L 436.2,218.5 L 434.1,243.3 L 476.8,245.4 L 476.6,242.1 L 475.9,241.2 L 475.5,239.7 L 475.5,238.2 L 475.8,237.3 L 475.3,236.6 L 475.3,231.7 L 473.7,227.3 L 473.9,226.1 L 473.6,225.3 L 473.8,224.6 L 473.6,224.6 L 473.8,224.2 L 473.6,223.9 L 474.0,222.9 L 473.5,221.8 L 473.3,220.3 L 463.7,220.1 Z" fill="#fff7ec" stroke="#fff" stroke-width="0.7"><title>North Dakota: 2 (explicit campuses 1; building-inferred sites 1; standalone points 0)</title></path><path d="M 566.7,311.4 L 568.0,310.6 L 570.1,308.0 L 569.0,308.1 L 568.6,307.3 L 568.4,307.5 L 567.8,307.2 L 567.6,306.5 L 566.2,305.3 L 566.4,304.8 L 565.0,303.5 L 565.3,302.6 L 564.9,301.1 L 563.3,300.2 L 562.9,298.9 L 560.4,300.5 L 558.9,299.8 L 558.0,300.6 L 557.4,300.4 L 556.6,299.7 L 554.4,299.6 L 553.6,298.0 L 552.8,297.7 L 552.6,297.2 L 551.6,297.7 L 550.7,297.2 L 549.9,298.0 L 550.4,298.7 L 550.2,299.2 L 550.8,299.4 L 550.7,300.2 L 549.6,300.4 L 548.5,301.3 L 547.8,300.9 L 546.8,301.1 L 547.1,302.7 L 546.0,303.6 L 545.7,304.8 L 544.7,305.1 L 544.3,306.0 L 544.3,307.3 L 543.7,307.9 L 542.1,307.2 L 542.0,306.5 L 541.4,306.1 L 541.7,306.6 L 540.9,306.8 L 541.1,307.2 L 540.5,307.6 L 540.7,308.4 L 540.2,308.5 L 540.0,309.2 L 539.8,308.7 L 539.3,308.8 L 538.7,308.0 L 537.4,308.9 L 537.0,310.0 L 534.9,308.8 L 534.3,309.2 L 533.7,308.7 L 533.6,309.2 L 533.9,309.5 L 533.5,309.9 L 533.3,309.3 L 532.3,309.6 L 531.8,309.3 L 531.6,309.7 L 531.9,310.2 L 531.6,310.6 L 531.1,310.4 L 530.4,311.6 L 531.1,313.0 L 528.5,314.0 L 528.3,314.9 L 529.0,315.9 L 528.8,316.6 L 528.1,316.6 L 525.7,315.6 L 524.7,316.1 L 524.2,317.2 L 524.8,317.9 L 524.3,319.0 L 524.8,319.5 L 524.2,319.8 L 524.5,320.3 L 524.1,321.2 L 523.2,320.7 L 523.0,321.8 L 531.9,321.1 L 531.6,319.7 L 533.1,319.6 L 533.1,319.9 L 541.9,318.9 L 560.1,317.4 L 561.0,316.6 L 563.5,315.6 L 563.8,314.6 L 565.0,314.2 L 565.0,313.4 L 565.7,313.0 L 565.7,312.1 L 566.7,311.4 Z M 521.9,321.4 L 522.2,321.9 L 522.6,321.6 L 522.2,321.2 L 521.9,321.4 Z" fill="#fff7ec" stroke="#fff" stroke-width="0.7"><title>Kentucky: 5 (explicit campuses 2; building-inferred sites 3; standalone points 0)</title></path><path d="M 594.0,288.8 L 594.0,288.3 L 593.5,288.1 L 593.6,287.5 L 593.2,287.6 L 593.2,287.3 L 592.8,287.2 L 593.1,286.7 L 592.4,286.7 L 592.4,287.0 L 590.8,286.4 L 590.4,287.1 L 589.4,287.3 L 589.6,287.5 L 589.2,287.6 L 589.6,287.9 L 589.3,288.2 L 587.9,288.2 L 587.3,287.9 L 587.5,287.7 L 587.2,287.5 L 586.4,289.4 L 585.4,289.2 L 585.2,290.0 L 583.4,291.9 L 582.8,287.6 L 576.3,288.7 L 575.1,281.2 L 574.5,281.5 L 574.3,281.9 L 574.9,282.7 L 574.8,283.4 L 575.1,283.9 L 574.5,286.0 L 574.6,287.4 L 574.3,287.4 L 574.5,287.8 L 574.1,288.7 L 574.4,289.1 L 574.3,289.9 L 572.5,292.2 L 571.5,292.7 L 571.0,292.2 L 570.4,292.9 L 570.4,293.5 L 569.7,293.6 L 569.4,294.4 L 569.5,295.0 L 569.1,295.2 L 569.7,296.5 L 569.2,296.4 L 568.8,297.0 L 568.5,296.1 L 567.8,295.9 L 566.9,298.0 L 567.4,299.5 L 566.8,299.8 L 566.7,300.9 L 565.0,301.4 L 565.3,302.6 L 565.0,303.5 L 566.4,304.8 L 566.2,305.3 L 566.8,305.6 L 567.0,306.2 L 567.6,306.5 L 567.8,307.2 L 568.4,307.5 L 568.6,307.3 L 569.0,308.1 L 570.4,308.2 L 570.0,308.6 L 571.1,310.0 L 571.7,310.0 L 572.3,310.5 L 573.1,310.3 L 574.2,309.1 L 575.2,309.8 L 576.9,309.0 L 577.4,308.5 L 577.3,307.8 L 577.9,308.2 L 579.2,307.1 L 579.7,307.5 L 580.7,306.6 L 580.5,306.2 L 581.1,305.5 L 580.5,305.1 L 581.1,303.5 L 581.9,302.3 L 582.2,300.7 L 582.9,299.8 L 582.7,299.5 L 583.2,299.0 L 583.3,297.1 L 584.1,297.3 L 584.6,298.0 L 585.9,298.1 L 587.0,294.3 L 587.9,294.8 L 588.3,293.7 L 589.3,292.8 L 590.2,291.0 L 589.9,290.7 L 590.3,289.4 L 590.1,288.5 L 593.8,290.6 L 594.2,288.9 L 594.0,288.8 Z" fill="#fff7ec" stroke="#fff" stroke-width="0.7"><title>West Virginia: 1 (explicit campuses 0; building-inferred sites 0; standalone points 1)</title></path><path d="M 598.0,405.1 L 598.5,403.9 L 597.6,393.2 L 593.7,385.9 L 592.1,383.5 L 591.5,382.0 L 591.8,380.4 L 591.2,379.3 L 587.8,375.4 L 585.0,370.7 L 582.7,364.6 L 582.7,363.2 L 580.8,363.4 L 578.3,362.8 L 577.8,363.8 L 578.3,365.1 L 578.3,366.6 L 577.4,366.9 L 576.7,365.5 L 576.8,365.2 L 558.2,366.5 L 556.9,364.2 L 538.8,366.1 L 538.7,367.2 L 540.4,368.7 L 540.2,370.0 L 540.9,370.6 L 540.0,371.6 L 540.4,371.7 L 539.9,371.9 L 540.0,373.2 L 545.6,371.7 L 548.5,371.7 L 550.7,372.3 L 552.9,373.5 L 553.9,374.2 L 554.4,375.7 L 555.2,376.6 L 556.5,376.5 L 558.0,377.1 L 560.4,375.8 L 562.2,374.0 L 563.3,373.7 L 563.9,372.4 L 564.5,372.1 L 566.7,373.1 L 567.4,374.2 L 568.3,374.7 L 568.8,375.9 L 570.2,376.7 L 571.7,379.0 L 572.5,379.3 L 573.5,379.0 L 573.9,380.6 L 574.7,382.0 L 574.5,383.9 L 573.8,384.9 L 574.2,387.9 L 575.2,389.3 L 575.5,391.0 L 579.6,396.3 L 581.0,399.1 L 582.2,399.8 L 583.4,399.7 L 584.9,403.2 L 585.9,404.1 L 587.3,404.0 L 588.6,404.9 L 589.5,406.1 L 589.6,407.4 L 590.1,408.4 L 591.2,409.3 L 588.8,409.7 L 586.1,411.8 L 583.8,412.3 L 583.0,413.7 L 583.9,413.6 L 584.3,414.2 L 585.0,413.9 L 586.3,414.7 L 587.2,414.4 L 587.1,413.8 L 587.9,414.2 L 589.0,413.6 L 589.1,412.9 L 592.4,411.8 L 596.5,408.6 L 598.0,405.1 Z M 577.1,414.1 L 577.4,414.9 L 578.1,415.3 L 578.0,414.4 L 580.6,414.0 L 580.6,413.5 L 579.9,412.6 L 578.7,412.5 L 577.5,413.2 L 577.1,414.1 Z" fill="#fee8c8" stroke="#fff" stroke-width="0.7"><title>Florida: 15 (explicit campuses 0; building-inferred sites 11; standalone points 4)</title></path><path d="M 531.6,306.3 L 532.0,306.3 L 532.5,305.2 L 532.4,304.7 L 533.0,304.3 L 532.9,304.0 L 533.2,303.6 L 533.1,303.2 L 533.8,302.3 L 533.4,301.3 L 533.6,300.6 L 532.5,299.1 L 533.0,298.4 L 532.6,297.7 L 533.1,297.3 L 531.4,277.4 L 533.3,277.3 L 533.9,271.1 L 512.2,272.7 L 512.2,273.0 L 513.6,274.0 L 513.8,274.9 L 515.2,275.7 L 515.4,277.4 L 515.3,278.3 L 514.5,278.9 L 514.4,280.1 L 512.6,281.3 L 510.2,281.8 L 509.9,283.3 L 511.0,284.4 L 510.9,285.8 L 510.2,286.7 L 510.1,288.0 L 508.4,289.0 L 508.6,290.3 L 508.2,290.6 L 507.9,291.7 L 508.0,293.3 L 508.9,295.8 L 513.1,299.5 L 513.5,300.8 L 513.3,301.2 L 514.0,302.5 L 514.5,302.6 L 515.1,301.8 L 517.2,302.7 L 516.7,303.8 L 516.9,304.7 L 515.8,307.0 L 516.0,308.0 L 517.7,309.5 L 518.9,309.9 L 518.6,310.2 L 518.8,310.6 L 519.4,310.4 L 520.6,311.2 L 520.7,311.6 L 521.6,312.0 L 522.0,312.8 L 521.7,313.3 L 522.4,314.5 L 521.9,315.4 L 522.3,315.6 L 522.9,317.3 L 523.5,317.7 L 523.7,317.5 L 523.3,317.0 L 523.7,317.0 L 524.2,317.8 L 524.5,317.7 L 524.2,317.0 L 525.4,315.6 L 528.1,316.6 L 528.8,316.6 L 529.0,315.9 L 528.3,314.9 L 528.6,313.9 L 531.1,313.0 L 530.4,311.6 L 531.1,310.4 L 530.7,310.3 L 531.1,310.1 L 530.6,309.6 L 531.1,309.6 L 530.8,309.4 L 531.1,309.0 L 530.8,308.3 L 531.4,308.0 L 531.0,307.9 L 531.6,307.3 L 531.0,306.6 L 531.6,306.3 Z" fill="#fc8d59" stroke="#fff" stroke-width="0.7"><title>Illinois: 38 (explicit campuses 2; building-inferred sites 33; standalone points 3)</title></path><path d="M 483.7,220.4 L 473.3,220.5 L 473.5,221.8 L 474.0,222.9 L 473.6,223.9 L 473.8,224.2 L 473.6,224.6 L 473.8,224.6 L 473.6,225.3 L 473.9,226.1 L 473.7,227.3 L 475.3,231.7 L 475.3,236.6 L 475.8,237.3 L 475.5,238.2 L 475.5,239.7 L 475.9,241.2 L 476.6,242.1 L 476.8,244.1 L 476.7,246.3 L 475.1,248.0 L 476.0,249.6 L 477.0,249.9 L 477.4,250.6 L 477.3,265.4 L 508.3,264.7 L 508.0,263.7 L 508.0,262.4 L 506.8,260.6 L 505.9,260.4 L 504.2,259.1 L 503.6,257.8 L 502.0,257.2 L 501.4,256.3 L 500.1,256.2 L 498.6,254.8 L 498.8,253.2 L 498.5,252.2 L 498.8,251.8 L 498.7,250.3 L 499.3,249.1 L 498.7,248.2 L 497.9,248.0 L 497.9,246.8 L 498.4,246.5 L 498.8,245.4 L 500.4,244.3 L 500.8,244.3 L 501.1,243.8 L 500.9,239.0 L 501.4,239.1 L 501.4,238.6 L 501.9,238.3 L 502.5,238.6 L 509.9,233.3 L 513.8,233.2 L 516.0,227.2 L 513.8,227.5 L 512.4,226.6 L 509.0,227.0 L 508.4,225.8 L 508.2,225.7 L 506.2,227.2 L 504.5,227.6 L 504.5,227.0 L 503.7,227.0 L 503.6,226.3 L 502.3,226.1 L 501.7,225.1 L 500.6,225.2 L 500.3,225.5 L 500.6,226.1 L 500.0,226.3 L 499.4,225.3 L 499.5,224.8 L 498.1,224.4 L 498.5,224.1 L 498.5,223.8 L 496.7,223.1 L 495.0,223.1 L 493.9,223.5 L 493.9,223.9 L 492.1,224.2 L 491.8,223.2 L 489.7,223.1 L 489.4,222.7 L 488.5,222.8 L 487.4,222.4 L 487.1,222.0 L 487.2,221.3 L 486.4,217.7 L 485.7,217.4 L 484.6,217.3 L 484.6,220.4 L 483.7,220.4 Z" fill="#fee8c8" stroke="#fff" stroke-width="0.7"><title>Minnesota: 9 (explicit campuses 1; building-inferred sites 8; standalone points 0)</title></path><path d="M 606.4,287.0 L 605.4,283.2 L 582.8,287.6 L 583.5,291.9 L 585.2,290.0 L 585.4,289.2 L 586.4,289.4 L 587.3,287.5 L 587.5,287.7 L 587.4,288.0 L 588.7,288.3 L 589.3,288.2 L 589.6,287.9 L 589.2,287.6 L 589.6,287.5 L 589.4,287.3 L 590.4,287.1 L 590.8,286.4 L 592.4,287.0 L 592.4,286.7 L 593.0,286.7 L 592.8,287.2 L 593.2,287.3 L 593.2,287.6 L 593.6,287.5 L 593.5,288.0 L 594.0,288.3 L 593.9,288.8 L 595.2,288.8 L 596.0,289.3 L 595.7,290.1 L 596.2,290.6 L 597.2,290.6 L 598.5,291.3 L 598.9,290.7 L 599.9,291.4 L 599.3,292.4 L 599.4,293.0 L 598.9,293.3 L 599.0,293.7 L 598.3,294.2 L 598.0,295.3 L 598.5,296.3 L 599.9,295.5 L 600.8,297.0 L 603.0,297.0 L 603.8,297.9 L 605.8,298.7 L 606.8,297.9 L 608.5,297.9 L 609.4,297.4 L 609.4,297.0 L 612.2,296.1 L 612.6,293.6 L 612.6,292.4 L 608.2,293.3 L 606.4,287.0 Z" fill="#fee8c8" stroke="#fff" stroke-width="0.7"><title>Maryland: 13 (explicit campuses 0; building-inferred sites 8; standalone points 5)</title></path><path d="M 629.0,260.8 L 628.0,260.1 L 627.3,258.4 L 624.9,259.1 L 626.1,263.9 L 625.7,264.9 L 626.7,265.7 L 626.5,265.0 L 630.1,262.8 L 630.3,262.6 L 629.5,260.9 L 629.0,260.8 Z M 627.4,265.7 L 627.9,266.1 L 628.4,265.9 L 628.3,264.8 L 627.6,264.6 L 627.4,265.7 Z" fill="#e5e7eb" stroke="#fff" stroke-width="0.7"><title>Rhode Island: no records</title></path><path d="M 369.9,208.5 L 366.3,207.8 L 361.4,228.7 L 361.9,230.4 L 361.4,231.0 L 361.8,232.6 L 362.0,233.3 L 363.2,234.2 L 363.4,235.4 L 361.8,237.5 L 361.1,238.9 L 360.2,239.7 L 360.0,240.7 L 359.3,241.5 L 358.5,241.8 L 356.9,244.2 L 356.9,245.2 L 358.1,245.8 L 358.4,246.5 L 357.8,247.0 L 357.9,247.8 L 357.0,249.0 L 353.8,263.7 L 389.3,270.5 L 392.5,250.4 L 391.6,249.4 L 391.7,249.0 L 390.9,247.8 L 390.0,248.6 L 390.2,249.5 L 388.9,249.1 L 388.1,249.4 L 387.9,248.9 L 386.5,249.0 L 385.5,248.5 L 385.1,248.6 L 384.8,249.3 L 382.6,248.6 L 382.0,249.6 L 381.2,248.7 L 381.0,245.9 L 380.4,245.4 L 379.8,245.6 L 379.3,244.9 L 379.1,244.2 L 379.5,244.1 L 379.6,243.3 L 378.5,241.4 L 378.2,240.1 L 378.5,239.2 L 378.1,239.2 L 378.4,238.5 L 377.8,238.3 L 377.8,237.7 L 377.3,237.6 L 376.8,238.3 L 375.8,238.5 L 375.2,239.1 L 374.6,238.2 L 374.0,238.1 L 374.2,237.4 L 374.7,237.0 L 374.4,236.3 L 374.8,235.8 L 375.5,235.6 L 375.6,234.9 L 375.1,234.3 L 375.5,233.8 L 375.2,233.4 L 375.7,233.3 L 375.7,232.5 L 376.0,232.4 L 376.1,231.6 L 376.5,231.2 L 376.4,230.7 L 376.8,230.6 L 377.1,229.6 L 375.5,229.4 L 375.4,228.5 L 374.7,228.7 L 374.7,228.0 L 374.2,227.7 L 374.0,227.2 L 374.2,226.7 L 373.6,226.2 L 373.3,225.0 L 372.6,224.2 L 372.6,223.6 L 371.7,223.2 L 371.4,222.4 L 370.5,221.8 L 371.3,221.5 L 370.7,220.8 L 371.1,220.5 L 371.1,219.6 L 369.8,217.0 L 371.6,208.9 L 369.9,208.5 Z" fill="#fff7ec" stroke="#fff" stroke-width="0.7"><title>Idaho: 3 (explicit campuses 1; building-inferred sites 2; standalone points 0)</title></path><rect x="440" y="548" width="22" height="14" fill="#e5e7eb"/><text x="466" y="560" font-size="10">No records</text><rect x="520" y="548" width="22" height="14" fill="#fff7ec"/><text x="546" y="560" font-size="10">1–5</text><rect x="594" y="548" width="22" height="14" fill="#fee8c8"/><text x="620" y="560" font-size="10">6–15</text><rect x="668" y="548" width="22" height="14" fill="#fdbb84"/><text x="694" y="560" font-size="10">16–30</text><rect x="742" y="548" width="22" height="14" fill="#fc8d59"/><text x="768" y="560" font-size="10">31–60</text><rect x="816" y="548" width="22" height="14" fill="#e34a33"/><text x="842" y="560" font-size="10">61–120</text><rect x="890" y="548" width="22" height="14" fill="#b30000"/><text x="916" y="560" font-size="10">121+</text><g id="state-abbreviation-labels" font-family="Arial, Helvetica, sans-serif" font-size="10.5" font-weight="700" fill="#111827" stroke="#ffffff" stroke-width="2.6" paint-order="stroke" stroke-linejoin="round" pointer-events="none" aria-label="state abbreviations"><text data-state="AL" x="544" y="353" text-anchor="middle" dominant-baseline="middle">AL</text><text data-state="AK" x="245" y="475" text-anchor="middle" dominant-baseline="middle">AK</text><text data-state="AZ" x="373" y="334" text-anchor="middle" dominant-baseline="middle">AZ</text><text data-state="AR" x="505" y="337" text-anchor="middle" dominant-baseline="middle">AR</text><text data-state="CA" x="333" y="295" text-anchor="middle" dominant-baseline="middle">CA</text><text data-state="CO" x="419" y="299" text-anchor="middle" dominant-baseline="middle">CO</text><text data-state="FL" x="574" y="391" text-anchor="middle" dominant-baseline="middle">FL</text><text data-state="GA" x="567" y="349" text-anchor="middle" dominant-baseline="middle">GA</text><text data-state="HI" x="350" y="511" text-anchor="middle" dominant-baseline="middle">HI</text><text data-state="ID" x="373" y="240" text-anchor="middle" dominant-baseline="middle">ID</text><text data-state="IL" x="521" y="294" text-anchor="middle" dominant-baseline="middle">IL</text><text data-state="IN" x="541" y="293" text-anchor="middle" dominant-baseline="middle">IN</text><text data-state="IA" x="496" y="278" text-anchor="middle" dominant-baseline="middle">IA</text><text data-state="KS" x="465" y="306" text-anchor="middle" dominant-baseline="middle">KS</text><text data-state="KY" x="546" y="310" text-anchor="middle" dominant-baseline="middle">KY</text><text data-state="LA" x="512" y="368" text-anchor="middle" dominant-baseline="middle">LA</text><text data-state="ME" x="635" y="231" text-anchor="middle" dominant-baseline="middle">ME</text><text data-state="MI" x="542" y="256" text-anchor="middle" dominant-baseline="middle">MI</text><text data-state="MN" x="495" y="241" text-anchor="middle" dominant-baseline="middle">MN</text><text data-state="MS" x="522" y="354" text-anchor="middle" dominant-baseline="middle">MS</text><text data-state="MO" x="503" y="307" text-anchor="middle" dominant-baseline="middle">MO</text><text data-state="MT" x="403" y="230" text-anchor="middle" dominant-baseline="middle">MT</text><text data-state="NE" x="458" y="281" text-anchor="middle" dominant-baseline="middle">NE</text><text data-state="NV" x="351" y="292" text-anchor="middle" dominant-baseline="middle">NV</text><text data-state="NM" x="412" y="337" text-anchor="middle" dominant-baseline="middle">NM</text><text data-state="NY" x="601" y="255" text-anchor="middle" dominant-baseline="middle">NY</text><text data-state="NC" x="587" y="322" text-anchor="middle" dominant-baseline="middle">NC</text><text data-state="ND" x="456" y="232" text-anchor="middle" dominant-baseline="middle">ND</text><text data-state="OH" x="562" y="284" text-anchor="middle" dominant-baseline="middle">OH</text><text data-state="OK" x="462" y="333" text-anchor="middle" dominant-baseline="middle">OK</text><text data-state="OR" x="337" y="241" text-anchor="middle" dominant-baseline="middle">OR</text><text data-state="PA" x="592" y="276" text-anchor="middle" dominant-baseline="middle">PA</text><text data-state="SC" x="581" y="340" text-anchor="middle" dominant-baseline="middle">SC</text><text data-state="SD" x="455" y="259" text-anchor="middle" dominant-baseline="middle">SD</text><text data-state="TN" x="546" y="325" text-anchor="middle" dominant-baseline="middle">TN</text><text data-state="TX" x="452" y="366" text-anchor="middle" dominant-baseline="middle">TX</text><text data-state="UT" x="382" y="290" text-anchor="middle" dominant-baseline="middle">UT</text><text data-state="VA" x="586" y="303" text-anchor="middle" dominant-baseline="middle">VA</text><text data-state="WA" x="344" y="215" text-anchor="middle" dominant-baseline="middle">WA</text><text data-state="WV" x="580" y="296" text-anchor="middle" dominant-baseline="middle">WV</text><text data-state="WI" x="517" y="253" text-anchor="middle" dominant-baseline="middle">WI</text><text data-state="WY" x="411" y="265" text-anchor="middle" dominant-baseline="middle">WY</text><line data-state="VT" x1="615.5" y1="245.1" x2="661" y2="220" stroke="#4b5563" stroke-width="0.8"/><text data-state="VT" x="666" y="220" text-anchor="start" dominant-baseline="middle">VT</text><line data-state="NH" x1="624.2" y1="243.1" x2="661" y2="236" stroke="#4b5563" stroke-width="0.8"/><text data-state="NH" x="666" y="236" text-anchor="start" dominant-baseline="middle">NH</text><line data-state="MA" x1="626.2" y1="257.3" x2="661" y2="252" stroke="#4b5563" stroke-width="0.8"/><text data-state="MA" x="666" y="252" text-anchor="start" dominant-baseline="middle">MA</text><line data-state="RI" x1="627.6" y1="262.2" x2="661" y2="268" stroke="#4b5563" stroke-width="0.8"/><text data-state="RI" x="666" y="268" text-anchor="start" dominant-baseline="middle">RI</text><line data-state="CT" x1="620.5" y1="264.6" x2="661" y2="284" stroke="#4b5563" stroke-width="0.8"/><text data-state="CT" x="666" y="284" text-anchor="start" dominant-baseline="middle">CT</text><line data-state="NJ" x1="611.5" y1="279.1" x2="661" y2="300" stroke="#4b5563" stroke-width="0.8"/><text data-state="NJ" x="666" y="300" text-anchor="start" dominant-baseline="middle">NJ</text><line data-state="DE" x1="609.0" y1="287.7" x2="661" y2="316" stroke="#4b5563" stroke-width="0.8"/><text data-state="DE" x="666" y="316" text-anchor="start" dominant-baseline="middle">DE</text><line data-state="MD" x1="597.7" y1="290.9" x2="661" y2="332" stroke="#4b5563" stroke-width="0.8"/><text data-state="MD" x="666" y="332" text-anchor="start" dominant-baseline="middle">MD</text></g></svg>

### State-level inferred independent-site table

| State | Abbreviation | Explicit campuses | Building-inferred sites | Standalone points | Total inferred sites |
|---|---|---:|---:|---:|---:|
| Virginia | VA | 31 | 132 | 3 | 166 |
| Texas | TX | 10 | 81 | 2 | 93 |
| California | CA | 3 | 69 | 9 | 81 |
| Arizona | AZ | 8 | 30 | 15 | 53 |
| Oregon | OR | 6 | 42 | 2 | 50 |
| Washington | WA | 10 | 36 | 1 | 47 |
| New Jersey | NJ | 4 | 38 | 3 | 45 |
| Ohio | OH | 9 | 35 | 1 | 45 |
| Illinois | IL | 2 | 33 | 3 | 38 |
| Nevada | NV | 3 | 28 | 2 | 33 |
| Georgia | GA | 7 | 20 | 3 | 30 |
| New York | NY | 0 | 16 | 4 | 20 |
| Colorado | CO | 0 | 16 | 3 | 19 |
| Iowa | IA | 5 | 12 | 0 | 17 |
| North Carolina | NC | 5 | 8 | 4 | 17 |
| Florida | FL | 0 | 11 | 4 | 15 |
| Tennessee | TN | 5 | 8 | 2 | 15 |
| Nebraska | NE | 5 | 9 | 0 | 14 |
| Maryland | MD | 0 | 8 | 5 | 13 |
| Missouri | MO | 1 | 10 | 2 | 13 |
| Wisconsin | WI | 1 | 9 | 3 | 13 |
| New Mexico | NM | 1 | 5 | 5 | 11 |
| Michigan | MI | 0 | 8 | 2 | 10 |
| Minnesota | MN | 1 | 8 | 0 | 9 |
| Pennsylvania | PA | 0 | 8 | 1 | 9 |
| Utah | UT | 1 | 7 | 1 | 9 |
| Kansas | KS | 0 | 7 | 0 | 7 |
| Alabama | AL | 3 | 3 | 0 | 6 |
| Oklahoma | OK | 0 | 4 | 2 | 6 |
| Wyoming | WY | 4 | 2 | 0 | 6 |
| Connecticut | CT | 0 | 4 | 1 | 5 |
| Indiana | IN | 0 | 3 | 2 | 5 |
| Kentucky | KY | 2 | 3 | 0 | 5 |
| Idaho | ID | 1 | 2 | 0 | 3 |
| Massachusetts | MA | 0 | 3 | 0 | 3 |
| New Hampshire | NH | 0 | 3 | 0 | 3 |
| South Carolina | SC | 2 | 1 | 0 | 3 |
| Arkansas | AR | 0 | 2 | 0 | 2 |
| Maine | ME | 0 | 1 | 1 | 2 |
| Mississippi | MS | 1 | 1 | 0 | 2 |
| Montana | MT | 0 | 2 | 0 | 2 |
| North Dakota | ND | 1 | 1 | 0 | 2 |
| South Dakota | SD | 0 | 2 | 0 | 2 |
| Louisiana | LA | 0 | 0 | 1 | 1 |
| West Virginia | WV | 0 | 0 | 1 | 1 |

Totals use the default decision boundaries. Explicit campuses, building-inferred sites, and standalone points are mutually exclusive `site_id` types; buildings and points already assigned to campuses are not counted again.

Virginia remains strongly clustered. Subsequent work must use spatial block cross-validation, leave-region-out evaluation, and time-based holdouts rather than randomly splitting adjacent sites.


# Part 2: Model Features Aligned with IM3

The earlier version listed 82 power, 83 water, and 106 land candidate fields. Many are not consistently available nationwide and do not enter the actual IM3/CERF-DC siting calculations. This version retains only variables that directly affect IM3 feasibility, cost, or market-gravity outcomes.

Four selection rules are applied:

1. The variable must appear in the inputs or calculation logic of the IM3 Atlas, the IM3 projected dataset, or CERF-DC.
2. It must be directly collectable or computable from nationwide public geospatial data, government statistics, or project-specific documents.
3. Derived scores, existing-facility identity, and future outcomes must not be fed back as inputs.
4. Hard exclusions are applied first and cannot be offset by high values on other model dimensions.

Growth scenarios, subjective weights, and composite scale values without a defined measurement basis are not observable site facts and are excluded from the core features.

**The map is retained, but state-level counts are not model inputs.** State, county, operator, facility name, and counts of existing facilities are used only for mapping, spatial grouping, and sample review.


## 2.1 Core model inputs: 15 features

<!-- CORE_FEATURES_START -->
| Measured meaning | Model feature | Unit or encoding | IM3 alignment | Availability |
|---|---|---|---|---|
| Total area required by the proposed campus | `campus_size_square_ft` | ft² | CERF-DC determines the number of contiguous feasible cells; the OSTI output uses the same field | A: project plan, permit filing, or OSTI project field |
| Rated power required by servers and other IT equipment | `data_center_it_power_mw` | MW | Input to CERF-DC energy, cost, and market-gravity calculations | A: project plan, utility interconnection request, or OSTI project field |
| Ratio of total facility energy to IT-equipment energy | `data_center_pue` | ratio | CERF-DC converts IT load to facility energy | B: project design, energy disclosure, or operating record; do not fabricate when missing |
| Land acquisition or assessed cost per unit area | `land_cost_per_sqft` | USD/ft² | Land-cost raster in CERF-DC locational cost | B: IM3 raster, county parcel assessment, or transaction record |
| Electricity charge per unit consumed | `electricity_rate_per_kwh` | USD/kWh | CERF-DC annual electricity-cost input | A/B: EIA or utility tariff; normalize to one base year |
| Local property-tax rate applicable to servers and other equipment | `personal_property_tax_rate` | decimal | CERF-DC equipment property-tax input | B: published state and local tax rates |
| Local real-property tax rate applicable to land and buildings | `real_property_tax_rate` | decimal | CERF-DC land and building property-tax input | B: published state and local tax rates |
| Sales-tax rate applicable to equipment purchases or electricity | `sales_tax_rate` | decimal | CERF-DC equipment and energy sales-tax input | B: published state and local tax rates |
| Distance to the nearest substation | `interconnection_distance_km` | km | CERF-DC interconnection-cost input; IM3 uses 2 km as a feasibility threshold | A: calculated from HIFLD/GeoPlatform substation coordinates |
| Rated voltage class of nearby transmission lines | `transmission_voltage_class` | kV or voltage class | IM3 Atlas retains HIFLD `VOLT_CLASS` | A: HIFLD/GeoPlatform transmission-line attribute |
| Number of providers offering symmetric 1 Gbps business broadband near the site | `fiber_provider_count_1gbps` | provider count | Atlas aggregates FCC business records with both directions ≥1 Gbps by H3 cell | A: FCC Broadband Data Collection |
| Minimum distance to a public water-supply service area | `municipal_water_distance_km` | km; 0 inside a service area | Atlas uses USGS Public Supply Service Areas; IM3 threshold is 5 km | A: calculated from USGS WSA v1 boundaries |
| Distance to the nearest Census market or population center | `market_distance_km` | km | CERF-DC gravity score uses distance from the candidate cell to the nearest market | B: IM3 market raster or Census market-center coordinates |
| Fraction of project cooling load served by water-based cooling | `water_cooling_fraction` | 0–1 | Direct CERF-DC input determining electricity use, water withdrawal, and water consumption | B: project design, water permit, or environmental-review filing; do not fabricate when missing |
| Contiguous buildable land remaining after all hard constraints | `feasible_contiguous_area_sqft` | ft² | CERF-DC selects only locations that pass constraints and provide sufficient contiguous area | A/B: connected-component calculation from the IM3 suitability raster |
<!-- CORE_FEATURES_END -->

**Availability:** A denotes a clearly defined nationwide public layer or verifiable project field. B denotes information that is obtainable in practice but requires local data, project documents, geospatial processing, or unit harmonization. Retain missing values as missing rather than substituting subjective scenarios or weights.

To avoid exact collinearity, retain only `water_cooling_fraction` among the core inputs. If cooling has only two categories, then `mechanical_cooling_fraction = 1 - water_cooling_fraction` and should not be entered again.


## 2.2 IM3 hard constraints: rule execution, not ordinary scoring features

| ID | IM3 feasibility rule | Proposed quantitative field | Treatment |
|---|---|---|---|
| H1 | Within 300 m of a federal airport runway | `airport_runway_distance_m` | Exclude directly if `<300` |
| H2 | Waterbody | `waterbody_overlap_pct` | Exclude the overlapping portion if `>0` |
| H3 | Slope above 16% | `max_slope_pct` | Exclude cells with values `>16` |
| H4 | Sinkhole-susceptible area | `sinkhole_susceptible` | Exclude if true |
| H5 | High coastal or inland flood risk | `high_flood_risk` | Exclude or require manual review if true |
| H6 | Parks, recreation areas, and cemeteries | `park_leisure_cemetery_overlap_pct` | Exclude the overlapping portion if `>0` |
| H7 | More than 2 km from a substation | `interconnection_distance_km` | Exclude if `>2` |
| H8 | More than 5 km from a municipal water service area | `municipal_water_distance_km` | Exclude if `>5` |
| H9 | More than 2 km from a high-speed fiber service area | `high_speed_fiber_distance_km` | Exclude if `>2` |
| H10 | PAD-US protected land | `padus_overlap_pct` | Exclude the overlapping portion if `>0` |
| H11 | Rail, primary-road, and secondary-road rights-of-way | `transport_right_of_way_overlap_pct` | Exclude the overlapping portion if `>0` |
| H12 | Military areas and training ranges | `military_overlap_pct` | Exclude the overlapping portion if `>0` |
| H13 | NLCD developed land and proximity restriction | `developed_land_overlap_pct`, `developed_land_distance_km` | Exclude developed cells; exclude sites more than `0.8 km` from developed land |

These fields belong to candidate generation and feasibility screening. If the research objective is to predict why a site was excluded, construct a separate rule label; do not mix exclusion with the post-screening recommendation grade in a compensatory total score.


## 2.3 Fields excluded from model inputs

| Field type | Examples | Correct use | Reason for exclusion from inputs |
|---|---|---|---|
| IM3 composite outcomes | `total_cost_million_usd`, `normalized_locational_cost`, `normalized_gravity_score`, `weighted_siting_score` / `total_weighted_siting_score` | Transparent rule baseline, auxiliary label, or model comparator | Calculated from core inputs; feeding them back causes target leakage |
| Derived cooling quantities | `cooling_energy_demand_mwh`, `cooling_water_demand_mgy`, `cooling_water_consumption_mgy` | Impact analysis or auxiliary output | Computable from IT MW, PUE, and cooling fraction; repeated input increases collinearity |
| Location and identity | `id`, `geometry`, `region`, `state`, `county`, `operator`, `name` | Spatial joins, grouped validation, and result display | A model may memorize states, markets, or operators instead of learning siting mechanisms |
| Existing-facility attributes | IM3 `sqft`, `type`, and state facility counts | Campus-size calibration, introductory map, or weak-label construction | Available only for built facilities; direct input treats the outcome itself as a cause |
| Project-count outcomes | `n_sites`, built/not-built status | Scenario demand or supervised label | A scenario outcome rather than an intrinsic site condition |

If the final target is a multiclass recommendation grade, `weighted_siting_score` may help initialize grade definitions or serve as a comparator, but it cannot also appear in the training features `X`.


# Part 3: Independent Outcome-Label Datasets and Feasibility

**Evidence status (August 27, 2026): search-incomplete.** No public source found in this review is a ready-to-train national table containing unique U.S. projects, final positive/negative decisions, decision dates, reasons, and all 15 historical site features. The feasible strategy is to combine public project inventories, event trackers, IM3 locations, and official local decision documents.

## 3.1 Available datasets and their roles

| Source | Best role | Main limitation | Recommended treatment |
|---|---|---|---|
| FracTracker Open U.S. Data Centers Tracker | Common primary project table for proposed, approved, operating, suspended, and cancelled projects | Status is not a verified causal outcome label; fields are incomplete | Use as the master candidate inventory and retain every source link |
| Tracking American AI Data Center Buildout / DataCenterTracker | Dated local actions, opposition, permit denials, withdrawals, and source documents | Rows are actions, not unique projects or final outcomes | Convert actions into project event histories, then verify the final decision |
| IM3 Existing Atlas | Historical operating-location supplement and spatial cross-check | Point, building, and campus layers can describe the same physical site | Apply the documented entity resolution and require independent operating evidence |
| Data Center Watch | Confirmed and blocked-project leads and market context | Public extracts are not a complete research table | Use for candidate discovery and triangulation |
| Aterio | Commercial U.S. facility and development coverage | Access and redistribution restrictions | Use only if research and redistribution rights are secured |
| Shovels.ai | Permit-based project discovery and construction signals | Commercial API; a permit record is not necessarily a final siting outcome | Use for candidate discovery and historical permit timing |
| UVA Data Center Policy Database | State and local policy context | Policy records are contextual, not project outcomes | Join by jurisdiction and time only as contextual evidence |
| dcmap.us Policy Tracker | Policy and moratorium context | Not a national project outcome table | Use for jurisdiction-time controls and document discovery |

**Recommended backbone:** FracTracker supplies the common project table; DataCenterTracker supplies events, timing, and reported reasons; IM3 supplements operating locations and supports cross-checks; official planning, zoning, utility, court, or permit documents determine strict labels.

## 3.2 Verified scale and field completeness

The FracTracker public sheet snapshot contained **1,665 parseable rows**, while its dashboard displayed approximately **1,677** records. The sheet included 740 Proposed, 528 Operating, 173 Approved/Permitted/Under construction, 67 Expanding, 69 Suspended, 78 Cancelled, and 10 Pre-proposal records. Field coverage was uneven: 1,308 rows had high location confidence, 655 had MW, 1,001 had acreage, 311 had project cost, 72 had cooling source, 85 had cooling type, 318 indicated pushback, and 1,548 contained at least one source link.

DataCenterTracker exposed **1,486 action records** at review time. Twelve mentioned `permit_denial`, 40 mentioned `project_withdrawal`, 325 included MW, 536 included acreage, and 899 identified a company or project; all records were source-linked. These are action counts, not counts of unique denied projects. The site supports CSV/JSON export and declares CC BY 4.0 for its export.

## 3.3 Strict outcome labels

| Label | Operational definition | Model use |
|---|---|---|
| `positive_strict` | Dated official evidence of final approval, permit issuance, construction, or operation | `y=1` |
| `negative_strict` | Dated final government, planning, zoning, or permit denial that was not later overturned | `y=0` |
| `withdrawn` | Applicant withdrew before a final adverse decision | Separate competing outcome; exclude from the first binary model |
| `cancelled_other` | Project cancelled for financing, corporate strategy, utility, land, or unknown reasons | Separate outcome; do not automatically label as `0` |
| `delayed_or_suspended` | Moratorium, litigation, review, or temporary pause without a final decision | Censored/pending outcome |
| `pending` | No final decision by the observation cutoff | Exclude from final-label training |

If a denial is later overturned and approval is issued, the final label is positive and the denial remains in the event history. A reported rejection reason may be an explanatory or factor-specific label, but it cannot also be used as an input for predicting that same outcome.

## 3.4 Recommended integration workflow

1. Freeze dated snapshots and record checksums, access dates, licenses, and source URLs.
2. Build one master project table with stable `project_id`, developer, project name, coordinates, jurisdiction, and source aliases.
3. Normalize MW, acreage, cost, dates, coordinates, status vocabulary, and missing-value codes.
4. Resolve entities with name, developer, address, parcel, coordinates, and source-link evidence; retain match confidence.
5. Create a separate event table containing event type, event date, reporting date, source, authority, and appeal/supersession links.
6. Assign strict labels only after two-person or adjudicated review of official evidence.
7. Define prediction time `t0` and join only feature values observable before `t0`.
8. Construct geographically and temporally matched positive/negative samples to reduce reporting and market imbalance.
9. Use nested spatial-temporal validation and a locked geographic test set.
10. Audit entity matches, label agreement, missingness, state/year coverage, and source-dependent selection bias.

Do not auto-label `Cancelled`, `Suspended`, or `Not approved` as negative; do not count all IM3 points, buildings, and campuses as separate successes; do not use post-decision features; and do not use the presence or number of news/source links as a predictor.

## 3.5 Next actions

1. Snapshot and hash the FracTracker sheet, its data dictionary, DataCenterTracker JSON, and the IM3 GeoPackage.
2. Process FracTracker Operating, Approved/Permitted/Under construction, Cancelled, and Suspended records first.
3. Manually review the 147 Cancelled plus Suspended candidates as leads, not automatic negatives.
4. Match DataCenterTracker `permit_denial`, `project_withdrawal`, and related actions to those projects.
5. Add IM3 sites only when independent evidence confirms operation.
6. Pilot the workflow in Arizona, Virginia, Georgia, Florida, and Indiana before national expansion.
7. Report strict-negative yield, state/year coverage, feature missingness, and sampled entity-match accuracy.
8. Join the 15 time-aligned model features only after label construction is frozen.

## 3.6 Feasibility and continuation gates

| Deliverable | Feasibility | Main condition |
|---|---|---|
| National candidate project table | High | Entity resolution and versioned snapshots |
| Strict positive labels | Medium-high | Independent approval/construction/operation evidence |
| Strict negative labels | Medium | Official final decisions and appeal tracking |
| Decision date and reason | Medium | Manual document review and controlled vocabulary |
| Fifteen historical features at `t0` | Medium-low to medium | Archived local, utility, and geospatial data |
| First interpretable ML benchmark | Medium-high | Sufficient strict negatives and spatial coverage |
| Deep-learning main result | Low initially | Much larger labeled sample and stable missingness patterns |

The following are planning gates, not a statistical power calculation: at least 100 strict negatives across at least 20 states and three decision years supports a national spatial-generalization experiment; 50–99 supports a matched case-control and interpretable-ML study labeled exploratory; fewer than 50 should prioritize dataset/label methodology and case studies rather than deep learning.


## 3.7 Dataset, download, and documentation URLs

- FracTracker: [project page](https://fractracker.org/data-centers/), [dashboard](https://experience.arcgis.com/experience/5a4d072ad01449bba5698a80103fb909), [dashboard documentation](https://experience.arcgis.com/experience/5a4d072ad01449bba5698a80103fb909/page/About), [public data sheet](https://docs.google.com/spreadsheets/d/1JJ6kcVo-NjlAYtznwHOki2DVl4WWV6lhy-eXhFCdKKU/edit?gid=386766486#gid=386766486), [data dictionary](https://docs.google.com/spreadsheets/d/1KVab4niu8X_GGne3WyyH_CdTP9Bt4uJSe6iK0fu9C48/edit?usp=sharing), and [terms](https://fractracker.org/terms-of-service/).
- DataCenterTracker: [site](https://datacentertracker.org/), [raw JSON](https://datacentertracker.org/data/fights.json), and [application/export logic](https://datacentertracker.org/js/app.js).
- IM3: [MSD-LIVE record](https://data.msdlive.org/records/65g71-a4731), [Atlas repository](https://github.com/IMMM-SFA/datacenter-atlas), [GeoPackage](https://github.com/IMMM-SFA/datacenter-atlas/blob/main/data_center_database/im3_us_data_center_locations.gpkg), [OSTI projected-locations record](https://www.osti.gov/biblio/2571680), and [CERF-DC repository](https://github.com/IMMM-SFA/cerf_data_centers).
- Additional project leads: [Data Center Watch](https://www.datacenterwatch.org/report), [Aterio U.S. overview](https://www.aterio.io/insights/us-data-centers), [Aterio dataset](https://www.aterio.io/datasets/lst_us_data_centers), [Shovels API](https://www.shovels.ai/solutions/api), and [Shovels permit analysis](https://www.shovels.ai/blog/data-center-permits-decisions/).
- Policy and local evidence: [UVA Data Center Policy Database](https://www.datacenterpolicy.com/), [UVA DTD Lab](https://dtdlab.virginia.edu/project/data-center-policies/), [dcmap.us Policy Tracker](https://dcmap.us/insights/policy/), [Piedmont Environmental Council map](https://www.pecva.org/region/loudoun/existing-and-proposed-data-centers-a-web-map/), and ArcGIS items [92b9f1d09ad54d9caa2bb195dd7a7312](https://www.arcgis.com/home/item.html?id=92b9f1d09ad54d9caa2bb195dd7a7312) and [440dfc2a2c954b398492a39f8de3c5cb](https://www.arcgis.com/home/item.html?id=440dfc2a2c954b398492a39f8de3c5cb).

**License boundary:** DataCenterTracker declares CC BY 4.0 for its export and IM3 identifies ODbL terms. FracTracker provides download, attribution, and noncommercial guidance alongside separate website terms; confirm the applicable terms before redistributing a full snapshot. Commercial sources should remain coverage checks or discovery aids unless explicit research and redistribution rights are obtained.
